In [ ]:
# %% [markdown]
# # Bengali LLM Hallucination Detection — Phase-2 submission notebook
# # FINAL Phase-1 pipeline (final_entry, public LB 0.954)
#
# TWO EXECUTION MODES, auto-detected from the mounted test set. Nothing to configure.
#
#   A. PUBLIC-RERUN MODE — reproduces the SELECTED Phase-1 submission (final_entry) EXACTLY.
#      Triggered when the mounted test set's id-set equals the attached reference
#      predictions CSV. Cell 5 loads the precomputed per-layer artifacts from the
#      attached Kaggle dataset "mdmeheduzzaman/bengali-halluc-v23-artifacts",
#      reassembles final_entry with the exact shipped router, and Cell 19 diffs row-by-row
#      and prints PASS / n_mismatches. Every heavy stage is skipped (runtime ~2 min).
#
#   B. HELD-OUT MODE (~5,000 unseen rows) — every layer is RECOMPUTED LIVE, offline.
#      Source matching (squad_bn/TyDiQA, bangla-mmlu, livemcq + sibling-site banks),
#      the deterministic Bengali-grammar tiers, the symbolic math solver, the
#      idiom/dictionary gloss tiers, the wiki-article tier, and finally the LLM
#      judges (Qwen3-8B thinking for the ctx residual and the math router; an
#      optional Qwen3-32B 4-bit logprob judge for the closed-book residual).
#
# ---------------------------------------------------------------------------
# THE final_entry LAYER STACK (measured on the 2,516 public test rows). Precedence is
# top-to-bottom within each segment; every layer is offline-reproducible.
#
#   reroute overrides   cb_gram 7 | ctx_gold3 2
#   leak                11     test (prompt,response) seen verbatim in the labeled split
#   -- context rows --
#   ctx_bio             1      Bengali ordinal-suffix date normalization rescue
#   ctx_gold            922    matched to squad_bn / TyDiQA-GoldP-bn gold answers
#   ctx_LAST_BIPARIT    7      বিপরীত-শব্দ canon
#   ctx_LAST_GLOSS      6      dictionary-gloss sweep
#   ctx_deterministic   191    সমাস / সন্ধি / উপসর্গ / বাগধারা rule tiers
#   ctx_think           223    Qwen3-8B thinking judge (+14B/logprob 2-of-3 arbitration)
#   -- closed-book rows --
#   cb_LAST_SWEEP       10     final gloss/canon sweep
#   cb_LAST_GLOSS       30     dictionary gloss
#   cb_mathsolve        139    symbolic math solver (math_solve.py, 26 templates)
#   cb_last             47     residual lookup tier (math / public-source / canon)
#   cb_livemcq2         711    livemcq MCQ bank fingerprint match
#   cb_gold             20     hishab/bangla-mmlu gold match
#   cb_math             11     Qwen3-8B thinking math router
#   cb_wiki             62     bn.wikipedia article grounding
#   cb_tail             106    idiom tail-4 gloss overlap
#   cb_wikisource       7      Bangla Academy dictionary gloss
#   cb_sites            1      sibling exam-site QA banks
#   cb_32B              2      Qwen3-32B 4-bit logprob judge @ P(halluc) > 0.50
#   cb_default          0      residual -> 0 (hallucinated)
#
#   TOTAL 2,516.  halluc(0)=1161  faithful(1)=1355.
#
# NOTE ON VERSIONING: work/build_final.py in the repo has since gained two more
# tiers (ctx_BANK 10, ctx_FINAL_ABS 2 — "the later unsubmitted build"), built after the daily submission
# quota ran out and therefore NEVER SUBMITTED. This notebook reproduces the
# SUBMITTED final_entry, i.e. source_match_ctx_bank.json and
# source_match_final_abstentions.json are deliberately EXCLUDED from the router
# (see EXCLUDED_ARTIFACTS below). Verified locally: reassembling with those two files
# reproduces the on-disk later-build CSV 2516/2516, and excluding them yields final_entry, which
# differs from the later unsubmitted build on exactly 14 rows.
#
# ---------------------------------------------------------------------------
# NO MODEL WAS FINE-TUNED. The "checkpoint" is open-weight Qwen3 (Apache-2.0)
# plus deterministic match indices built from public datasets. The "training
# notebooks" are the index builders and threshold-fitting scripts listed in
# work/kaggle/README.md. Nothing in this pipeline was fitted to leaderboard
# feedback; the only fitted scalars are two judge thresholds fitted on the
# released labeled sample split (ctx logprob 0.19, 32B logprob 0.50).
#
# PHASE-2 COMPLIANCE
#   * Runs fully offline: HF_HUB_OFFLINE / TRANSFORMERS_OFFLINE are set in Cell 2
#     and every input is a mounted Kaggle dataset. No network call anywhere.
#   * REQUIRED KAGGLE DATASET ATTACHMENTS (exact ids):
#       mdmeheduzzaman/bengali-halluc-v23-artifacts   27 router artifacts +
#                                                     submission_reference.csv
#                                                     (PUBLIC-RERUN mode)
#       mdmeheduzzaman/bengali-halluc-tier-code       the verbatim tier modules +
#                                                     assets/ wiki_articles/
#                                                     source_data/ (HELD-OUT mode)
#       mdmeheduzzaman/bengali-halluc-source-banks    livemcq_qa2.json,
#                                                     sites_<domain>_qa.json x12,
#                                                     wikt_pages.json, idiom/grammar
#                                                     KBs (HELD-OUT mode)
#     plus the competition data, the QA source datasets (squad_bn, bangla-mmlu)
#     and the Qwen3 weights. Cell 5 / Cell 6c print exactly what resolved.
#   * Single Kaggle kernel, GPU = P100 or 2xT4.
#   * Weights budget (Phase-2 cap 50 GB): Qwen3-8B 4-bit ~5 GB on disk (or 16.4 GB
#     fp16 shards quantized at load) + optional Qwen3-32B 4-bit ~18 GB
#     = at most ~34.4 GB < 50 GB. PASS. Models are loaded SEQUENTIALLY with
#     del + gc.collect() + torch.cuda.empty_cache() between stages (Cell 2
#     free_model), so peak VRAM never holds both.
#   * Runtime: 7.5 h soft budget, 8.25 h hard fuse, 9 h platform limit. The
#     RuntimeGovernor (Cell 3) measures s/row after a warmup, projects the whole
#     run, and degrades the judge ladder thinking -> logprob -> substring rule.
#   * A VALID submission.csv is emitted at every milestone, so a crash or a
#     wall-clock kill can never produce a DNF.



In [ ]:
# %% ------------------------------------------------------------------
# Cell 1: Config
import os, sys, csv, json, glob, gc, math, time, random, re, collections, unicodedata

NOTEBOOK_T0 = time.time()

# --- runtime budget (seconds) ---
KAGGLE_LIMIT_H = 9.0          # hard platform limit
SOFT_BUDGET_H = 7.5           # governor degrades on a projected breach of this
HARD_CAP_H = 8.25             # stop all LLM scoring here; rule-fill the rest
SOFT_BUDGET_S = SOFT_BUDGET_H * 3600
HARD_CAP_S = HARD_CAP_H * 3600

SEED = 42
BATCH_SIZE = 16               # halved automatically on CUDA OOM
MAX_TOKENS = 3072             # tokenizer cap (left-truncate: keep QA + instruction tail)

# --- judge model budgets ---
# Thinking-mode generation is the expensive path: measured 30-45 s/row for the
# ctx judge (450 new tokens) and up to ~60 s/row for the math judge (2048 new
# tokens, long chains). The governor's budget model uses THINK_SROW_EST until it
# has measured the real rate on this hardware.
THINKING_MODE_ENABLED = True
MATH_MAX_NEW_TOKENS = 2048    # work/math_router.py budget
CTX_MAX_NEW_TOKENS = 450      # work/ctx_think.py budget
THINK_SROW_EST = 38.0         # s/row prior for thinking mode (midpoint of 30-45)
LOGPROB_SROW_EST = 2.0        # s/row prior for the one-token logprob fallback
THINK_WARMUP_ROWS = 12        # rows judged before the first projection
THINK_RECHECK_EVERY = 25      # re-project every N rows during the thinking pass
THINK_PROJ_MARGIN_S = 300     # safety margin added to every projection

# --- ctx judge arbitration constants (the exact shipped final_entry values) ---
CTX_LOGPROB_THR = 0.19        # ctx_sub_judge.json P(halluc) -> 0/1, fitted on samples

# --- optional Qwen3-32B closed-book residual judge (final_entry cb_32B layer) ---
# Optional and fully degradable: if the 32B weights are not attached, or the
# governor projects a budget breach, the residual rows fall back to the 8B
# logprob judge and finally to the final_entry default (0 = hallucinated).
J32_ENABLED = True
J32_THR = 0.50                # P(halluc) threshold — the exact shipped constant
J32_MAX_TOKENS = 2048         # measured OOM fix: 3072 -> 2048
J32_BATCH_CB, J32_BATCH_CTX = 6, 2   # measured OOM fix: 24/8 -> 6/2
J32_LOAD_EST_S = 900          # conservative load estimate (18 GB shards on 2xT4)
J32_SROW_EST = 2.5            # pre-measurement s/row prior (measured 1.73 on 2xT4)
J32_EMPTY_CACHE_EVERY = 60    # periodic cache clear (rows) to fight fragmentation

MODEL_CANDIDATES = [          # Qwen3-8B; first hit wins
    "/kaggle/input/qwen-3/transformers/8b/1",
    "/kaggle/input/qwen-3/transformers/8b/2",
    "/kaggle/input/qwen3-8b/transformers/default/1",
    "/kaggle/input/qwen3-8b",
    "/kaggle/input/qwen3-8b-hf",
]
MODEL32_CANDIDATES = [        # Qwen3-32B (optional)
    "/kaggle/input/qwen-3/transformers/32b/1",
    "/kaggle/input/qwen-3/transformers/32b/2",
    "/kaggle/input/qwen3-32b/transformers/default/1",
    "/kaggle/input/qwen3-32b",
    "/kaggle/input/qwen3-32b-bnb-4bit",
]
LOCAL_MODEL_FALLBACK = os.environ.get("HALLU_MODEL", "Qwen/Qwen3-8B")  # dev only

# --- Phase-2 rerun protocol flags ---
REPRODUCE_CHECK = True        # diff the output against the attached reference CSV
MAKE_SCALE_TEST = False       # rehearsal: duplicate the public set to ~5,000 rows
FORCE_LIVE_PIPELINE = False   # debug: run the live pipeline even on the public set
REFERENCE_PRED_NAMES = [      # searched under /kaggle/input, first hit wins
    # NOTE: the published Kaggle dataset carries the reference under the name
    # "predictions_v23_reference.csv" -- it MUST stay first in this list, or
    # PUBLIC-RERUN mode cannot be detected and the reproduction diff is skipped.
    "predictions_v23_reference.csv",
    "submission_reference.csv", "reference_predictions.csv", "predictions.csv",
]

# --- final_entry precomputed layer artifacts -------------------------------------
# Kaggle dataset "mdmeheduzzaman/bengali-halluc-v23-artifacts" (27 artifacts +
# predictions_v23_reference.csv, all at the dataset ROOT under the basenames
# below). key -> filename. Located by BASENAME anywhere under /kaggle/input;
# local dev falls back to work/. Required only for PUBLIC-RERUN MODE; held-out
# never reads them. Cell 5 asserts that every non-optional entry resolved.
SUBMISSION_ARTIFACTS = {
    # --- context segment ---
    "ctx":            "source_match_ctx.json",                  # ctx gold (squad_bn/TyDiQA)
    "ctx3":           "source_match_ctx3.json",                  # ctx gold TyDiQA reroute block
    "ctx_bio":        "source_match_ctx_bio.json",               # ordinal-date rescue
    "ctx_grammar":    "source_match_ctx_grammar.json",           # সমাস / grammar canon
    "ctx_biparit":    "source_match_ctx_biparit_sandhi.json",    # সন্ধি + বিপরীত শব্দ
    "ctx_idiom_upa":  "source_match_ctx_idiom_upasarga.json",    # বাগধারা + উপসর্গ
    "last_biparit":   "source_match_last_biparit.json",          # final বিপরীত sweep
    "last_gloss":     "source_match_last_gloss.json",            # final gloss sweep (ctx + cb)
    "ctx_think":      "ctx_think_test.json",                     # Qwen3-8B thinking verdicts
    "ctx_14b":        "ctx_14b_test.partial",                    # Qwen3-14B arbitration (jsonl)
    "ctx_lp":         "ctx_sub_judge.json",                      # 8B logprob (3rd arbitration vote)
    # --- closed-book segment ---
    "last_sweep":     "source_match_last_sweep.json",            # final cb sweep
    "cb_mathsolve":   "source_match_cb_mathsolve.json",          # symbolic solver
    "cb_last":        "source_match_cb_last.json",               # residual lookup tier
    "cb_livemcq2":    "source_match_cb_livemcq2.json",           # livemcq bank
    "cb":             "source_match_cb.json",                    # bangla-mmlu gold
    "cb2":            "source_match_cb2.json",                   # bangla-mmlu expansion tiers
    "math":           "math_test.json",                          # 8B thinking math router
    "cb_wiki":        "source_match_cb_wiki.json",               # wiki-article grounding
    "cb_tail":        "source_match_cb_tail.json",               # idiom tail-4
    "wikt":           "wikt_test_pred.json",                     # bn.wiktionary gloss
    "cb_wikisource":  "source_match_cb_wikisource.json",         # Bangla Academy dictionary
    "cb_sites":       "source_match_cb_sites.json",              # sibling exam-site banks
    "cb_gram":        "source_match_cb_gram.json",               # bn_grammar_kb canon
    "cb_ocr":         "source_match_cb_ocr.json",                # OCR-recovered mmlu
    "j32":            "judge32b_scores.json",                    # Qwen3-32B logprob scores
    # --- router control ---
    "reroute":        "reroute.json",                            # per-row layer overrides
}
# Artifacts the router tolerates as absent (they only ever ADD coverage).
OPTIONAL_ARTIFACTS = {"ctx_bio", "cb_ocr", "cb_sites", "cb_gram", "ctx3", "reroute"}

# Files that exist in the repo but are DELIBERATELY EXCLUDED: they belong to a
# later build that was never submitted. Loading them would reproduce that later
# build, not the 0.954 submitted entry (they differ on 14 rows).
EXCLUDED_ARTIFACTS = ["source_match_ctx_bank.json", "source_match_final_abstentions.json"]

# --- offline data banks required by HELD-OUT MODE ------------------------
# Kaggle dataset "mdmeheduzzaman/bengali-halluc-source-banks". Each bank is
# optional (a missing bank disables exactly one tier and its rows fall through
# to the next layer — never a crash).
#
# NAME FLATTENING. Kaggle's dataset uploader drops subdirectories, so the banks
# were uploaded under FLAT basenames while the local dev repo keeps the nested
# layout the builders wrote:
#     source_hunt/livemcq/qa2.json      -> livemcq_qa2.json
#     source_hunt/sites/<domain>/qa.json -> sites_<domain>_qa.json   (x12)
# Both spellings are searched, flat first (Kaggle), nested second (local dev).
LIVEMCQ_BANK_NAMES = ["livemcq_qa2.json", "livemcq_qa.json",   # flat (Kaggle)
                      "qa2.json", "qa.json"]                   # nested (local dev)
SITE_BANK_PATTERNS = ["sites_*_qa.json",          # flat (Kaggle)
                      "sites/*/qa.json",          # nested (local dev / any mount)
                      "source_hunt/sites/*/qa.json"]
SITE_FLAT_RE = re.compile(r"^sites_(.+)_qa\.json$")   # -> <domain>
BANK_FUZZY_THRESH = 97.0                            # matcher.predict_row default

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:                                   # running as notebook cells
    _HERE = os.getcwd()
_WORK = os.path.abspath(os.path.join(_HERE, ".."))  # local dev repo layout: work/
LOCAL_DATA_DIR = os.path.join(_WORK, "..", "bengali-hallucination")
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SUBMISSION_PATH = os.path.join(OUT_DIR, "submission.csv")

SAMPLES_FILE = "dataset samples.json"
TEST_FILE = "test set.csv"


def find_input(name, local=None, required=True):
    """Locate a file by name under /kaggle/input (recursive), else a local dev path."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if hits:
        return hits[0]
    if local and os.path.exists(local):
        return local
    if required:
        raise FileNotFoundError(f"{name!r} not found under /kaggle/input (or {local})")
    return None


def find_inputs(patterns, local_roots=()):
    """ALL files matching any of `patterns` (glob, may contain '/') anywhere under
    /kaggle/input, else under the given local dev roots. Used for the multi-file
    banks whose Kaggle upload flattened `sites/<domain>/qa.json` into
    `sites_<domain>_qa.json` — both spellings are tried and de-duplicated."""
    hits = []
    for pat in patterns:
        hits += glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    if not hits:
        for root in local_roots:
            for pat in patterns:
                hits += glob.glob(os.path.join(root, "**", pat), recursive=True)
    return sorted(dict.fromkeys(p for p in hits if os.path.isfile(p)))


def find_dir(name, local=None):
    """Locate a directory by name under /kaggle/input, else a local dev path."""
    hits = sorted(p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)
                  if os.path.isdir(p))
    if hits:
        return hits[0]
    if local and os.path.isdir(local):
        return local
    return None




In [ ]:
# %% ------------------------------------------------------------------
# Cell 2: Determinism, offline environment, sequential model-load helpers
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["HF_HUB_OFFLINE"] = "1"          # competition rule: no internet at inference
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Measured 32B OOM fix: the expandable-segments allocator avoids the
# fragmentation OOMs seen with large 4-bit prefill batches. MUST be set before
# the first `import torch`, which happens two lines below.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"gpus={torch.cuda.device_count()}")
for _i in range(torch.cuda.device_count()):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  gpu{_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  cc={_p.major}.{_p.minor}")


def free_model(*names):
    """Requirement 4: release a model stage's VRAM before the next load.
    Deletes the named globals, drops references, collects, empties the CUDA
    cache. Called between the 8B and 32B stages so peak VRAM never holds both
    (2xT4 = 2x15 GB cannot hold 8B + 32B simultaneously)."""
    g = globals()
    for n in names:
        if g.get(n) is not None:
            del g[n]
            g[n] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb = sum(torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count()))
        print(f"[vram] freed {names} -> {free_gb/2**30:.1f} GiB free across "
              f"{torch.cuda.device_count()} gpu(s)", flush=True)


try:
    from rapidfuzz import fuzz as _rf_fuzz, process as _rf_process
    HAVE_RAPIDFUZZ = True
except ImportError:                          # degrade gracefully
    HAVE_RAPIDFUZZ = False
    print("WARNING: rapidfuzz unavailable -> fuzzy matching tiers disabled "
          "(exact / canonical-key tiers unaffected)")

import difflib


def _ratio(a, b):
    """0-100 similarity ratio; rapidfuzz if present, difflib otherwise."""
    if HAVE_RAPIDFUZZ:
        return _rf_fuzz.ratio(a, b)
    return 100.0 * difflib.SequenceMatcher(None, a, b).ratio()




In [ ]:
# %% ------------------------------------------------------------------
# Cell 3: RuntimeGovernor — judge-ladder degradation for the held-out fold
#
# THE HELD-OUT RISK. On the public set the source tiers cover 2,282 of 2,516
# rows, so only 234 rows ever reach an LLM judge and thinking mode fits easily.
# On an UNSEEN ~5,000-row fold the undisclosed sources may match far fewer rows,
# and the judge pool can grow by an order of magnitude. Thinking mode at
# ~38 s/row over 2,000 ctx rows would be 21 h — a guaranteed DNF.
#
# THE LADDER (cheapest quality loss first):
#   T0_thinking    Qwen3-8B thinking-mode generation, VERDICT-line parse.
#                  Measured 0.875 F1 on authored-ctx rows. ~30-45 s/row.
#   T1_logprob     Qwen3-8B one-token Yes/No prefill (no generation).
#                  Measured 0.792 on the same rows. ~1-3 s/row (~15x cheaper).
#   T2_substring   Parameter-free verbatim-substring rule on the context.
#                  Measured ~0.90 F1 on unmatched ctx rows; 0 s/row, no GPU.
#                  This rung is also the hard-cap fuse's fill, so the notebook
#                  ALWAYS produces a complete, valid CSV.
#
# The governor measures the real s/row after a warmup batch, re-projects every
# THINK_RECHECK_EVERY rows, and degrades on any projected breach of the 7.5 h
# soft budget. Pending non-ladder stages (the 32B judge) are reserved in the
# projection via extra_pending_s so the ladder degrades EARLY enough to leave
# room for them. The 8.25 h hard fuse trips regardless and rule-fills the rest.
class RuntimeGovernor:

    LEVELS = [
        ("T0_thinking",  dict(mode="thinking", srow=THINK_SROW_EST)),
        ("T1_logprob",   dict(mode="logprob",  srow=LOGPROB_SROW_EST)),
        ("T2_substring", dict(mode="rule",     srow=0.0)),
    ]

    def __init__(self, t0, soft_s, hard_s):
        self.t0 = t0
        self.soft_s = soft_s
        self.hard_s = hard_s
        self.level = 0
        self.tripped = False              # hard-cap fuse
        self.rate = {}                    # measured s/row per mode
        self.pending = collections.OrderedDict()   # stage -> rows still to judge
        self.extra_pending_s = 0.0        # reserved cost of non-ladder stages (32B)
        self.events = []                  # degradation log for the final report
        self._phase = None                # (stage, phase_t0, mode) being measured
        self._mark = (t0, 0)              # (wall clock, rows done) at its start

    # ---- level accessors -------------------------------------------------
    @property
    def cfg(self):
        return self.LEVELS[self.level][1]

    def level_key(self):
        return self.LEVELS[self.level][0]

    def mode(self):
        return self.cfg["mode"]

    def elapsed(self):
        return time.time() - self.t0

    # ---- planning --------------------------------------------------------
    def set_plan(self, **stage_rows):
        """Register the judge workload, e.g. set_plan(math=40, ctx_think=1800)."""
        for k, v in stage_rows.items():
            self.pending[k] = int(v)
        print(f"governor plan: {dict(self.pending)} rows to judge; "
              f"start level {self.level_key()}  "
              f"(soft {self.soft_s/3600:.2f}h / hard {self.hard_s/3600:.2f}h)")

    def consume(self, stage, n=1):
        if stage in self.pending:
            self.pending[stage] = max(0, self.pending[stage] - n)

    def srow(self, mode=None):
        """Best available s/row for a mode: measured > prior."""
        mode = mode or self.mode()
        if self.rate.get(mode):
            return self.rate[mode]
        for key, cfg in self.LEVELS:
            if cfg["mode"] == mode:
                return cfg["srow"]
        return 0.0

    def project(self, extra_rows=0):
        """Projected TOTAL notebook seconds if we finish everything pending at
        the CURRENT level, plus any reserved non-ladder stages."""
        rem = (sum(self.pending.values()) + extra_rows) * self.srow()
        return self.elapsed() + rem + self.extra_pending_s + THINK_PROJ_MARGIN_S

    # ---- degradation -----------------------------------------------------
    def _degrade_once(self, reason):
        old = self.level_key()
        self.level = min(self.level + 1, len(self.LEVELS) - 1)
        new = self.level_key()
        msg = f"DEGRADE {old} -> {new} ({reason})"
        print("\n" + "!" * 78, flush=True)
        print(f"!!! RUNTIME GOVERNOR: {msg}")
        print(f"!!! new judge mode: {self.mode()}  "
              f"(~{self.srow():.1f}s/row, {sum(self.pending.values())} rows pending)")
        if self.mode() == "rule":
            print("!!! all remaining judge rows now use the parameter-free "
                  "substring rule — no GPU, guarantees a complete CSV")
        print("!" * 78 + "\n", flush=True)
        self.events.append(msg)

    def checkpoint(self, stage, done, total, phase_t0, force=False):
        """Call after each judged row/batch. Returns 'ok' | 'degrade' | 'fuse'."""
        now = time.time()
        # RATE MEASUREMENT, PER MODE. The caller's phase_t0 marks when the LOOP
        # started, not when the CURRENT mode started. After a mid-loop degrade
        # (e.g. the math router flipping thinking -> logprob) the old mode's
        # seconds would be charged to the new one — measuring logprob at ~30
        # s/row instead of ~2 and triggering a spurious further degrade all the
        # way down to the substring rule. So the clock is re-marked whenever the
        # (stage, loop, mode) triple changes, and the always-free "rule" rung is
        # never assigned a rate (its srow must stay 0, or project() would charge
        # seconds to a rung that costs none).
        mode = self.mode()
        if self._phase != (stage, phase_t0, mode):
            self._phase, self._mark = (stage, phase_t0, mode), (now, done)
        elif mode != "rule":
            _t_ref, _d_ref = self._mark
            if done > _d_ref:
                self.rate[mode] = (now - _t_ref) / (done - _d_ref)
        if self.elapsed() > self.hard_s:
            self.tripped = True
            self.level = len(self.LEVELS) - 1
            print(f"  [{stage}] HARD CAP {self.hard_s/3600:.2f}h reached -> stopping "
                  f"all LLM scoring; remaining rows use the substring rule", flush=True)
            return "fuse"
        projected = self.project()
        flag = "  [WARN: projection exceeds soft budget]" if projected > self.soft_s else ""
        if force or done % max(THINK_RECHECK_EVERY, 1) == 0:
            print(f"  [{stage}] {done}/{total}  elapsed={self.elapsed()/60:.1f}m  "
                  f"{self.srow():.2f}s/row  level={self.level_key()}  "
                  f"projected_total={projected/3600:.2f}h{flag}", flush=True)
        if projected > self.soft_s and self.level < len(self.LEVELS) - 1:
            self._degrade_once(f"projected {projected/3600:.2f}h > soft budget "
                               f"{self.soft_s/3600:.2f}h")
            return "degrade"
        return "ok"


governor = RuntimeGovernor(NOTEBOOK_T0, SOFT_BUDGET_S, HARD_CAP_S)



In [ ]:
# %% ------------------------------------------------------------------
# Cell 4: Competition data loading, metric, crash-safe submission writer


def _clean(r):
    for k in ("prompt_bn", "response_bn", "context"):
        r[k] = "" if r.get(k) is None else str(r[k])
    if r["context"].strip() in ("[NULL]", ""):
        r["context"] = ""
    return r


# --- MOUNT INVENTORY -------------------------------------------------------
# Printed first so that ANY failure below can be diagnosed from the log alone:
# it shows exactly which datasets/models/competitions were attached, which is
# the single most common cause of a failed organiser rerun.
if os.path.isdir("/kaggle/input"):
    print("MOUNTS under /kaggle/input:")
    for _d in sorted(os.listdir("/kaggle/input")):
        try:
            _n = sum(len(fs) for _, _, fs in os.walk(f"/kaggle/input/{_d}"))
        except Exception:
            _n = -1
        print(f"   {_d}  ({_n} files)")
else:
    print("MOUNTS: /kaggle/input does not exist (local dev run)")

# The labelled sample split is OPTIONAL. The organisers have stated that
# "dataset samples.json" will be ABSENT from the held-out fold, so treating it
# as required would turn a missing file into a FileNotFoundError before any
# submission.csv exists -- i.e. a DNF, which is disqualifying. It is loaded
# when present (it powers the exact-leak layer and the per-tier validation
# printouts) and degrades to an empty split when not. Every consumer of S is
# guarded: the leak layer contributes 0 rows, validation gates print
# "unavailable" instead of a score, and the tier shim hands out an empty list.
# No prediction path depends on it.
samples_path = find_input(SAMPLES_FILE, os.path.join(LOCAL_DATA_DIR, SAMPLES_FILE),
                          required=False)
def resolve_test_csv():
    """Pick the RIGHT 'test set.csv' when more than one is mounted.

    WHY THIS EXISTS. The Phase-1 competition ('bengali-hallucination') and the
    Phase-2 evaluation competition ('iut-2026-datathon-phase-2-evaluation')
    BOTH ship a file called 'test set.csv'. If an organiser attaches both, a
    plain sorted-glob resolves to whichever sorts first -- and 'bengali-...'
    sorts before 'iut-...', so the notebook would silently score the PHASE-1
    rows and emit predictions for the wrong data. That is worse than crashing,
    because it fails quietly and looks like a valid run.

    Rule: prefer a mount whose path names the held-out/evaluation competition;
    otherwise fall back to the sorted-glob answer. Every candidate and the
    final choice are printed so the selection is auditable from the log.
    """
    cands = sorted(glob.glob(f"/kaggle/input/**/{TEST_FILE}", recursive=True))
    if not cands:
        local = os.path.join(LOCAL_DATA_DIR, TEST_FILE)
        if os.path.exists(local):
            return local
        # Name-based lookup failed. The held-out fold may ship the test file
        # under a different NAME (test.csv, test_set.csv, heldout.csv, ...).
        # Rather than DNF on a rename, sniff every mounted CSV for the known
        # schema and take the largest match. Losing the round to a filename
        # would be the most avoidable failure possible.
        want = {"id", "prompt_bn", "response_bn"}
        sniffed = []
        for p in glob.glob("/kaggle/input/**/*.csv", recursive=True):
            try:
                with open(p, encoding="utf-8", newline="") as fh:
                    cols = set(next(csv.reader(fh)))
            except Exception:
                continue
            if want <= cols:
                sniffed.append((os.path.getsize(p), p))
        if sniffed:
            sniffed.sort(reverse=True)
            print(f"NOTE: no file named {TEST_FILE!r}; sniffed the test schema "
                  f"{want} and matched {len(sniffed)} CSV(s).")
            print(f"   -> using {sniffed[0][1]}")
            return sniffed[0][1]
        return None
    if len(cands) > 1:
        print(f"NOTE: {len(cands)} '{TEST_FILE}' files are mounted:")
        for c in cands:
            print("   -", c)
    PREFER = ("phase-2", "phase2", "evaluation", "eval", "held-out", "heldout")
    for c in cands:
        low = c.lower()
        if any(k in low for k in PREFER):
            if len(cands) > 1:
                print(f"   -> chose {c} (path names the evaluation competition)")
            return c
    if len(cands) > 1:
        print(f"   -> no evaluation-named mount; falling back to {cands[0]}")
    return cands[0]


test_path = resolve_test_csv()
if test_path is None:
    raise FileNotFoundError(
        f"{TEST_FILE!r} not found under /kaggle/input. Attach the competition "
        f"dataset. (Mounted dirs: {sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'none'})")
print("samples:", samples_path or "ABSENT -> leak layer + validation disabled")
print("test   :", test_path)

if samples_path:
    with open(samples_path, encoding="utf-8") as f:
        S = [_clean(dict(r)) for r in json.load(f)]
    for r in S:
        r["label"] = int(r["label"])
else:
    S = []
HAVE_SAMPLES = bool(S)
with open(test_path, encoding="utf-8", newline="") as f:
    T = [_clean(dict(r)) for r in csv.DictReader(f)]

CTX_T = [i for i, r in enumerate(T) if r["context"]]
CB_T = [i for i, r in enumerate(T) if not r["context"]]
CTX_S = [i for i, r in enumerate(S) if r["context"]]
CB_S = [i for i, r in enumerate(S) if not r["context"]]
print(f"samples={len(S)} (ctx={len(CTX_S)} cb={len(CB_S)})  "
      f"test={len(T)} (ctx={len(CTX_T)} cb={len(CB_T)})")


def f1_halluc(y_true, y_pred):
    """Competition metric: binary F1 on the HALLUCINATED class (label == 0)."""
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    pr = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0


# --- DNF-safety: crash-safe progressive submission writer ---------------------
# Kaggle scores whatever submission.csv exists when the kernel ends; if none
# exists the run is a DNF (0 on the 50% Phase-2 component). So a VALID file is
# written up front and re-written at every milestone from whatever layers have
# completed. Each milestone strictly improves it. The final_entry default for an
# undecidable closed-book row is 0 (the shipped router's cb_default); an
# undecidable context row falls to the substring rule.
V23_CB_DEFAULT = 0


def _write_submission(pred):
    with open(SUBMISSION_PATH, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "label"])
        for r, p in zip(T, pred):
            w.writerow([r["id"], int(p)])


def emit_submission(tag=""):
    """Assemble submission.csv from whatever globals exist so far. On the
    public-rerun fast path the reassembled final_entry predictions are complete and
    authoritative — re-emit them verbatim."""
    G = globals()
    final_entry = G.get("final_pred") or {}
    if len(final_entry) == len(T):
        _write_submission([final_entry[i] for i in range(len(T))])
        print(f"[crash-safety] submission.csv re-written from the final_entry artifacts"
              f"{(' ' + tag) if tag else ''}: {len(T)} rows")
        return
    out, filled = [], 0
    for i, r in enumerate(T):
        p = live_lookup(i, r) if "live_lookup" in G else None
        if p is None:
            p = G.get("ctx_sub", {}).get(i) if r["context"] else None
            if p is None:
                p = 1 if r["context"] else V23_CB_DEFAULT
                filled += 1
        out.append(int(p))
    _write_submission(out)
    print(f"[crash-safety] submission.csv written{(' ' + tag) if tag else ''}: "
          f"{len(out)} rows, {filled} defaulted")


# BASELINE WRITE — the very first thing done after the test rows are in memory.
# The comment above says "a VALID file is written up front"; until this call it
# was not: the first emit_submission() sat ~1,100 lines downstream, after
# artifact resolution, the fast-path assembly and the whole tier bridge. ANY
# exception before that point left /kaggle/working with NO submission.csv at
# all — a DNF and a zero on the 50% Phase-2 component. Reproduced: a test CSV
# carrying the reference id-set in a different row order crashed the Cell-5
# assembly with a KeyError and produced no file whatsoever. Now the floor
# (every ctx row faithful, every cb row the shipped default) exists on disk
# before anything that can fail runs, and every later milestone overwrites it
# with something strictly better.
emit_submission(tag="(baseline floor, before any layer)")




In [ ]:
# %% ------------------------------------------------------------------
# Cell 4b: MAKE_SCALE_TEST — held-out-scale runtime rehearsal (OFF by default)
# Duplicates the public test set to ~5,000 rows (the organizers' stated held-out
# size) so a rehearsal exercises the RuntimeGovernor under realistic load.
if MAKE_SCALE_TEST:
    print("\n" + "#" * 78)
    print("### MAKE_SCALE_TEST: duplicating the test set for a runtime rehearsal ###")
    print("#" * 78 + "\n")
    _orig_n = len(T)
    _dup = []
    for _k, _r in enumerate(T):
        _rr = dict(_r)
        _rr["id"] = str(_orig_n + _k + 1)
        _dup.append(_rr)
    T = T + _dup
    CTX_T = [i for i, r in enumerate(T) if r["context"]]
    CB_T = [i for i, r in enumerate(T) if not r["context"]]
    print(f"scale test: {_orig_n} -> {len(T)} rows (ctx={len(CTX_T)} cb={len(CB_T)})")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 4c: Reference predictions + PUBLIC-RERUN detection.
# The organizers run this notebook twice. On the public test set it must
# reproduce the Phase-1 leaderboard predictions exactly. Attach the submitted
# final_entry CSV as a Kaggle dataset under any name in REFERENCE_PRED_NAMES.
REF_PRED_PATH = None
for _name in REFERENCE_PRED_NAMES:
    REF_PRED_PATH = find_input(
        _name, os.path.join(_WORK, _name), required=False)
    if REF_PRED_PATH:
        break
REF_PRED, REF_ORDER = None, None
if REF_PRED_PATH:
    with open(REF_PRED_PATH, encoding="utf-8", newline="") as f:
        _ref_rows = list(csv.DictReader(f))
    REF_PRED = {str(row["id"]).strip(): int(row["label"]) for row in _ref_rows}
    REF_ORDER = [str(row["id"]).strip() for row in _ref_rows]
    print(f"reference predictions: {REF_PRED_PATH} ({len(REF_PRED)} rows)")
else:
    print("reference predictions: none attached (held-out run, or diff disabled)")

IS_PUBLIC_RERUN = (bool(REPRODUCE_CHECK) and not MAKE_SCALE_TEST
                   and not FORCE_LIVE_PIPELINE
                   and REF_PRED is not None
                   and set(REF_PRED) == {str(r["id"]).strip() for r in T})
print(f"REPRODUCE_CHECK={REPRODUCE_CHECK}  MAKE_SCALE_TEST={MAKE_SCALE_TEST}  "
      f"-> IS_PUBLIC_RERUN={IS_PUBLIC_RERUN}")
if IS_PUBLIC_RERUN:
    print("  public test set detected: predictions MUST equal the submitted entry CSV")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 5: PUBLIC-RERUN MODE — reassemble final_entry from the precomputed artifacts.
#
# The submitted Phase-1 file is a DETERMINISTIC assembly of per-layer artifact
# files. When (a) this is the public rerun, (b) the mounted rows are in the same
# ORDER as the reference CSV (artifacts are keyed by row index), and (c) every
# required artifact is attached, we reassemble final_entry exactly and skip every heavy
# stage. Any missing precondition falls back to the full live pipeline.
#
# This is work/build_final.py's router ported verbatim, with the two later-build-only
# tiers removed (EXCLUDED_ARTIFACTS). Verified locally: 0/2516 mismatches vs the
# submitted CSV.
FAST_PATH = False
_seg_bad = []          # segment-consistency violations (see the gate below)
final_pred, final_layer = {}, {}

_art = {}
for _k, _fn in SUBMISSION_ARTIFACTS.items():
    _art[_k] = find_input(_fn, os.path.join(_WORK, _fn), required=False)
_missing = [SUBMISSION_ARTIFACTS[k] for k in SUBMISSION_ARTIFACTS
            if _art[k] is None and k not in OPTIONAL_ARTIFACTS]
_same_order = (REF_ORDER is not None
               and REF_ORDER == [str(r["id"]).strip() for r in T])

# ORDER TOLERANCE. Every final_entry artifact is keyed by the row's INDEX in the public
# test CSV, so the router needs a mapping from mounted row -> artifact index.
# When the mounted order equals the reference order that mapping is the
# identity (the shipped case, and the one that reproduced 0/2516). If the
# organizers hand back the same 2,516 ids in a DIFFERENT order, the identity
# mapping is wrong, and the previous code responded by disabling the fast path
# entirely — falling through to a live recompute that provably CANNOT reproduce
# the 14B/32B judge layers of final_entry. Instead, translate through the id: the
# assembly stays exact and the emitted CSV keeps the MOUNTED row order.
# Requires ids to be unique and to match the reference id-set 1:1.
_ids_T = [str(r["id"]).strip() for r in T]
_ref_index = {rid: k for k, rid in enumerate(REF_ORDER)} if REF_ORDER else {}
_id_addressable = (REF_ORDER is not None
                   and len(_ids_T) == len(set(_ids_T)) == len(REF_ORDER)
                   and set(_ids_T) == set(REF_ORDER))
# mounted row index -> artifact row index
ART_IDX = ([_ref_index[r] for r in _ids_T] if _id_addressable
           else list(range(len(T))))
if _id_addressable and not _same_order:
    print(f"NOTE: the mounted test rows carry the reference id-set but in a "
          f"DIFFERENT order. The final_entry artifacts are index-keyed, so the router "
          f"will address them through an id->index permutation "
          f"({sum(1 for a, b in enumerate(ART_IDX) if a != b)} rows displaced). "
          f"submission.csv keeps the mounted row order.")

# ARTIFACT RESOLUTION REPORT. The single highest Phase-2 risk is a stale or
# unattached artifacts dataset: Cell 5 would silently fall through to the live
# pipeline and the public rerun would not reproduce final_entry. So every entry is
# reported by name, and the reference CSV is reported alongside them.
_absent = sorted(SUBMISSION_ARTIFACTS[k] for k in SUBMISSION_ARTIFACTS if _art[k] is None)
print(f"final_entry artifacts: {len(SUBMISSION_ARTIFACTS) - len(_absent)}/{len(SUBMISSION_ARTIFACTS)} "
      f"resolved  (reference CSV: "
      f"{os.path.basename(REF_PRED_PATH) if REF_PRED_PATH else 'MISSING'})")
if _absent:
    print(f"  NOT FOUND ({len(_absent)}): {_absent}")
    print(f"    required-missing (blocks the fast path): {_missing or 'none'}")
    print(f"    optional-missing (coverage only): "
          f"{sorted(set(_absent) - set(_missing)) or 'none'}")
    print("  -> attach mdmeheduzzaman/bengali-halluc-v23-artifacts")
else:
    print("  all 27 artifacts + the reference CSV resolved from "
          + os.path.dirname(_art["ctx"]))

for _x in EXCLUDED_ARTIFACTS:
    if find_input(_x, required=False):
        print(f"NOTE: {_x} is attached but DELIBERATELY IGNORED — it belongs to a "
              f"later build that was never submitted; this notebook reproduces "
              f"the submitted entry.")


def _rows(path):
    """Read a tier artifact as a list of row dicts (list or {'rows': [...]})."""
    if not path or not os.path.exists(path):
        return []
    try:
        d = json.load(open(path, encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: unreadable artifact {path}: {e}")
        return []
    return d if isinstance(d, list) else d.get("rows", [])


def _imap(path, valid=(0, 1)):
    """{row_index: pred} from a tier artifact keyed by 'i'."""
    return {r["i"]: r["pred"] for r in _rows(path)
            if r.get("i") is not None and r.get("pred") in valid}


if IS_PUBLIC_RERUN and (_same_order or _id_addressable) and not _missing:
    FAST_PATH = True
    print("PUBLIC-RERUN MODE: all final_entry layer artifacts found -> reassembling the "
          "submitted final_entry predictions (every heavy stage will be skipped)")

    # ---- final-sweep tiers (highest precedence within their segment) ----
    _last_gloss = _imap(_art["last_gloss"])
    _last_bip = _imap(_art["last_biparit"])
    _last_sweep = _imap(_art["last_sweep"])

    # ---- deterministic Bengali-grammar block (first file wins per row) ----
    _det = {}
    for _key in ("ctx_grammar", "ctx_biparit", "ctx_idiom_upa"):
        for _r in _rows(_art[_key]):
            _det.setdefault(_r["i"], _r["pred"])

    _ctxbio = _imap(_art["ctx_bio"], valid=(0, 1))
    _mathsolve = {r["i"]: r["pred"] for r in _rows(_art["cb_mathsolve"])}
    _cblast = {r["i"]: r["pred"] for r in _rows(_art["cb_last"])}
    _cbtail = {r["i"]: r["pred"] for r in _rows(_art["cb_tail"])}
    _lm2 = {r["i"]: r["pred"] for r in _rows(_art["cb_livemcq2"])}
    _sites = {r["i"]: r["pred"] for r in _rows(_art["cb_sites"])}
    _gram = {r["i"]: r["pred"] for r in _rows(_art["cb_gram"])}
    _wsrc = {r["i"]: r["pred"] for r in _rows(_art["cb_wikisource"])}
    _wikt = {r["i"]: r["pred"] for r in _rows(_art["wikt"])}
    _ocr = {r["i"]: r["pred"] for r in _rows(_art["cb_ocr"])}
    _wiki = {r["i"]: r["pred"] for r in _rows(_art["cb_wiki"])}

    _reroute = json.load(open(_art["reroute"], encoding="utf-8")) if _art["reroute"] else {}
    _ctx3 = json.load(open(_art["ctx3"], encoding="utf-8")) if _art["ctx3"] else {}
    _j32 = {int(k): v for k, v in
            json.load(open(_art["j32"], encoding="utf-8"))["scores"].items()}

    # ---- ctx judge: 8B thinking, arbitrated 2-of-3 with 14B + 8B logprob ----
    _tk = json.load(open(_art["ctx_think"], encoding="utf-8"))
    _ctx_think = dict(zip(_tk["idx"], _tk["pred"]))
    _b14 = {}
    with open(_art["ctx_14b"], encoding="utf-8") as _f:
        for _line in _f:
            if _line.strip():
                _d = json.loads(_line)
                _b14[_d["i"]] = _d["p"]
    _cj = json.load(open(_art["ctx_lp"], encoding="utf-8"))
    _lp = {i: (0 if s > CTX_LOGPROB_THR else 1)
           for i, s in zip(_cj["idx"], _cj["scores"])}
    for _i, _p14 in _b14.items():
        if _i in _ctx_think:
            _votes = [_ctx_think[_i], _p14] + ([_lp[_i]] if _i in _lp else [])
            _ctx_think[_i] = 1 if sum(_votes) * 2 > len(_votes) else 0

    _ctx_match = json.load(open(_art["ctx"], encoding="utf-8"))
    _cbm = _rows(_art["cb"])
    _d2 = json.load(open(_art["cb2"], encoding="utf-8"))
    _cbm += _d2["rows"] if isinstance(_d2, dict) else _d2
    _cb_pred = {r["id"] - 1: r["pred"] for r in _cbm if r.get("pred") in (0, 1)}
    _mt = json.load(open(_art["math"], encoding="utf-8"))
    _math = dict(zip(_mt["idx"], _mt["pred"]))

    # ---- SEGMENT-CONSISTENCY GATE -------------------------------------------
    # IS_PUBLIC_RERUN is decided on the id SET alone, but every artifact is keyed
    # by ROW INDEX, so the fast path silently assumes that id N still carries the
    # SAME ROW CONTENT it carried in the public CSV. Nothing checked that.
    # Reproduced: a 2,516-row CSV with the reference id-set but shuffled content
    # entered the fast path and died with `KeyError: 0` in the assembly loop
    # below, before any submission.csv had ever been written — a DNF. And when
    # the segments happen to line up by luck the failure is WORSE than a crash:
    # the notebook would emit memorized public predictions against unrelated
    # rows, silently.
    #
    # The check: every artifact index carries a segment (a ctx artifact's keys
    # are context rows, a cb artifact's keys are closed-book rows). If the id ->
    # index mapping is sound, the segment of ART_IDX[i] must equal the segment of
    # mounted row i. A single mismatch proves the ids no longer address the rows
    # the artifacts were built from, and the fast path is abandoned in favour of
    # the live pipeline rather than trusted.
    _ctx_art_idx = set(_ctx_match) | {str(k) for k in _det} | {str(k) for k in _ctx_think}
    _cb_art_idx = (set(_mathsolve) | set(_cblast) | set(_lm2) | set(_cb_pred)
                   | set(_cbtail) | set(_wikt) | set(_wsrc) | set(_wiki))
    _seg_bad = []
    for _i, _r in enumerate(T):
        _a = ART_IDX[_i]
        _is_ctx = bool(_r["context"])
        if _is_ctx and _a in _cb_art_idx:
            _seg_bad.append((_i, _a, "ctx row -> closed-book artifact index"))
        elif not _is_ctx and str(_a) in _ctx_art_idx:
            _seg_bad.append((_i, _a, "closed-book row -> ctx artifact index"))
        if len(_seg_bad) >= 5:
            break
    if _seg_bad:
        FAST_PATH = False
        print("!" * 78)
        print("!!! PUBLIC-RERUN FAST PATH ABANDONED — SEGMENT-CONSISTENCY CHECK FAILED")
        print(f"!!! {len(_seg_bad)}+ mounted rows address an artifact index of the "
              f"OTHER segment, e.g.:")
        for _i, _a, _why in _seg_bad[:3]:
            print(f"!!!   mounted row {_i} (id={str(T[_i]['id']).strip()}) "
                  f"-> artifact index {_a}: {_why}")
        print("!!! The test CSV carries the reference id-set but NOT the reference "
              "row content, so the index-keyed artifacts do not describe these "
              "rows. Falling back to the full live pipeline.")
        print("!" * 78, flush=True)

if FAST_PATH:
    _byp = collections.defaultdict(list)
    for _r in S:
        _byp[_r["prompt_bn"].strip()].append(_r)

    _n23 = collections.Counter()
    for _i, _r in enumerate(T):
        # _i addresses the MOUNTED row (and the output); _a addresses the
        # index-keyed final_entry artifacts. Identical unless the rows came back
        # permuted — see ART_IDX above.
        _a = ART_IDX[_i]
        _rt = _reroute.get(str(_a))
        if _rt == "cb_gram" and _a in _gram:
            _p, _lay = _gram[_a], "cb_gram"
        elif (_rt == "ctx_gold_tydiqa" and str(_a) in _ctx3
              and _ctx3[str(_a)].get("pred_label") in (0, 1)):
            _p, _lay = _ctx3[str(_a)]["pred_label"], "ctx_gold3"
        else:
            _exact = [o for o in _byp.get(_r["prompt_bn"].strip(), [])
                      if str(o["response_bn"]).strip() == str(_r["response_bn"]).strip()]
            _m = _ctx_match.get(str(_a))
            if _exact:
                _p, _lay = _exact[0]["label"], "leak"
            elif _r["context"]:
                if _a in _ctxbio:
                    _p, _lay = _ctxbio[_a], "ctx_bio"
                elif _m and _m.get("pred_label") in (0, 1) and not _m.get("suspect_gold"):
                    _p, _lay = _m["pred_label"], "ctx_gold"
                # later-build-only rungs (ctx_BANK, ctx_FINAL_ABS) intentionally absent here
                elif _a in _last_bip:
                    _p, _lay = _last_bip[_a], "ctx_LAST_BIPARIT"
                elif _a in _last_gloss:
                    _p, _lay = _last_gloss[_a], "ctx_LAST_GLOSS"
                elif _a in _det:
                    _p, _lay = _det[_a], "ctx_deterministic"
                else:
                    _p, _lay = _ctx_think[_a], "ctx_think"
            elif _a in _last_sweep:
                _p, _lay = _last_sweep[_a], "cb_LAST_SWEEP"
            elif _a in _last_gloss:
                _p, _lay = _last_gloss[_a], "cb_LAST_GLOSS"
            elif _a in _mathsolve:
                _p, _lay = _mathsolve[_a], "cb_mathsolve"
            elif _a in _cblast:
                _p, _lay = _cblast[_a], "cb_last"
            elif _a in _lm2:
                _p, _lay = _lm2[_a], "cb_livemcq2"
            elif _a in _cb_pred:
                _p, _lay = _cb_pred[_a], "cb_gold"
            elif _a in _math:
                _p, _lay = _math[_a], "cb_math"
            elif _a in _wiki:
                _p, _lay = _wiki[_a], "cb_wiki"
            elif _a in _cbtail:
                _p, _lay = _cbtail[_a], "cb_tail"
            elif _a in _wikt:
                _p, _lay = _wikt[_a], "cb_idiom"
            elif _a in _wsrc:
                _p, _lay = _wsrc[_a], "cb_wikisource"
            elif _a in _sites:
                _p, _lay = _sites[_a], "cb_sites"
            elif _a in _ocr:
                _p, _lay = _ocr[_a], "cb_ocr"
            elif _a in _j32:
                _p, _lay = (0 if _j32[_a] > J32_THR else 1), "cb_32B"
            else:
                _p, _lay = V23_CB_DEFAULT, "cb_default"
        final_pred[_i] = int(_p)
        final_layer[_i] = _lay
        _n23[_lay] += 1

    print("final_entry layers:", dict(sorted(_n23.items())))
    print(f"final_entry distribution: halluc(0)={sum(1 for p in final_pred.values() if p == 0)}  "
          f"faithful(1)={sum(1 for p in final_pred.values() if p == 1)}")
    emit_submission(tag="(final_entry fast-path assembly)")
    _mism = [i for i, r in enumerate(T)
             if final_pred[i] != REF_PRED[str(r["id"]).strip()]]
    print("final_entry EARLY REPRODUCTION "
          + (f"PASS: all {len(T)} rows identical to the reference CSV" if not _mism
             else f"FAIL: n_mismatches = {len(_mism)}/{len(T)} (details in Cell 19)"))
else:
    _why = ("FORCE_LIVE_PIPELINE" if FORCE_LIVE_PIPELINE else
            "not the public rerun" if not IS_PUBLIC_RERUN else
            "test rows are not id-addressable against the reference CSV "
            "(duplicate ids, or the id-set differs)" if not _id_addressable else
            "the reference id-set is present but the rows behind those ids are "
            "NOT the public rows (segment-consistency check failed)"
            if _seg_bad else
            f"missing artifacts: {_missing}")
    print(f"public-rerun fast path DISABLED ({_why}) -> HELD-OUT MODE, full live "
          f"pipeline")
    if IS_PUBLIC_RERUN and not FORCE_LIVE_PIPELINE:
        print("  WARNING: a live recompute cannot bit-reproduce the 14B/32B judge "
              "layers of the submitted entry. Attach the artifacts dataset.")

LIVE = not FAST_PATH



In [ ]:
# %% ------------------------------------------------------------------
# Cell 6: Bengali text normalization (ports of work/bn_num.py, work/ctx_model.py
#         and the work/source_hunt_ctx.py norms — all parameter-free).
BN2ASCII = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

UNITS = {
    "শূন্য": 0, "এক": 1, "দুই": 2, "দু": 2, "তিন": 3, "চার": 4, "পাঁচ": 5,
    "পাচ": 5, "ছয়": 6, "সাত": 7, "আট": 8, "নয়": 9,
    "দশ": 10, "এগারো": 11, "এগার": 11, "বারো": 12, "বার": 12,
    "তেরো": 13, "তের": 13, "চৌদ্দ": 14, "চোদ্দ": 14, "পনেরো": 15, "পনের": 15,
    "ষোল": 16, "ষোলো": 16, "সতেরো": 17, "সতের": 17, "আঠারো": 18, "আঠার": 18,
    "উনিশ": 19, "ঊনিশ": 19, "বিশ": 20, "কুড়ি": 20, "একুশ": 21, "বাইশ": 22,
    "তেইশ": 23, "চব্বিশ": 24, "পঁচিশ": 25, "পচিশ": 25, "ছাব্বিশ": 26,
    "সাতাশ": 27, "আঠাশ": 28, "আটাশ": 28, "ঊনত্রিশ": 29, "উনত্রিশ": 29,
    "ত্রিশ": 30, "একত্রিশ": 31, "বত্রিশ": 32, "তেত্রিশ": 33, "চৌত্রিশ": 34,
    "পঁয়ত্রিশ": 35, "ছত্রিশ": 36, "সাঁইত্রিশ": 37, "আটত্রিশ": 38,
    "ঊনচল্লিশ": 39, "উনচল্লিশ": 39, "চল্লিশ": 40, "একচল্লিশ": 41,
    "বিয়াল্লিশ": 42, "তেতাল্লিশ": 43, "চুয়াল্লিশ": 44,
    "পঁয়তাল্লিশ": 45, "ছেচল্লিশ": 46, "সাতচল্লিশ": 47,
    "আটচল্লিশ": 48, "ঊনপঞ্চাশ": 49, "উনপঞ্চাশ": 49, "পঞ্চাশ": 50,
    "একান্ন": 51, "বাহান্ন": 52, "তিপ্পান্ন": 53, "চুয়ান্ন": 54, "পঞ্চান্ন": 55,
    "ছাপ্পান্ন": 56, "সাতান্ন": 57, "আটান্ন": 58, "ঊনষাট": 59, "উনষাট": 59,
    "ষাট": 60, "একষট্টি": 61, "বাষট্টি": 62, "তেষট্টি": 63, "চৌষট্টি": 64,
    "পঁয়ষট্টি": 65, "ছেষট্টি": 66, "সাতষট্টি": 67, "আটষট্টি": 68,
    "ঊনসত্তর": 69, "উনসত্তর": 69, "সত্তর": 70, "একাত্তর": 71, "বাহাত্তর": 72,
    "তিয়াত্তর": 73, "চুয়াত্তর": 74, "পঁচাত্তর": 75, "ছিয়াত্তর": 76,
    "সাতাত্তর": 77, "আটাত্তর": 78, "ঊনআশি": 79, "ঊনাশি": 79, "উনাশি": 79,
    "আশি": 80, "একাশি": 81, "বিরাশি": 82, "তিরাশি": 83, "চুরাশি": 84,
    "পঁচাশি": 85, "ছিয়াশি": 86, "সাতাশি": 87, "আটাশি": 88,
    "ঊননব্বই": 89, "উননব্বই": 89, "নব্বই": 90, "একানব্বই": 91,
    "বিরানব্বই": 92, "তিরানব্বই": 93, "চুরানব্বই": 94, "পঁচানব্বই": 95,
    "ছিয়ানব্বই": 96, "সাতানব্বই": 97, "আটানব্বই": 98, "নিরানব্বই": 99,
}
HUNDRED = {"শ", "শত", "শো"}
BIG = {"হাজার": 1000, "লক্ষ": 100000, "লাখ": 100000, "কোটি": 10000000}
BIG_HUNT = dict(BIG)
for _w, _v in [("মিলিয়ন", 10 ** 6), ("বিলিয়ন", 10 ** 9)]:
    BIG_HUNT[_w] = _v
    BIG_HUNT[unicodedata.normalize("NFC", _w)] = _v
COUNTERS = {"টি", "টা", "জন", "খানা", "খানি"}

NUM_TOKEN = re.compile(r"\d+(?:\.\d+)?|[^।,\.\-‐-―'\"“”‘’()!?;:\s\d]+")


def _kind(tok, big):
    if re.fullmatch(r"\d+(?:\.\d+)?", tok):
        return ("digit", float(tok) if "." in tok else int(tok))
    if tok in UNITS:
        return ("unit", UNITS[tok])
    if tok in HUNDRED:
        return ("H",)
    if tok in big:
        return ("B", big[tok])
    for h in ("শো", "শ"):
        if tok.endswith(h) and tok[:-len(h)] in UNITS:
            return ("unit", UNITS[tok[:-len(h)]] * 100)
    for c in COUNTERS:
        if tok.endswith(c) and tok[:-len(c)] in UNITS:
            return ("unit", UNITS[tok[:-len(c)]])
    return None


def _fmt(v):
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return f"#{v}#"


def canon_numbers(text, big=BIG):
    """Replace every maximal Bengali number-word/digit span with #<value>#, so
    'আঠারশ বত্রিশ' == '১৮৩২' == '1832'. Parameter-free canonicalization."""
    t = str(text).translate(BN2ASCII)
    t = re.sub(r"(?<=\d),(?=\d)", "", t)
    tokens = list(NUM_TOKEN.finditer(t))
    parts, pos, i = [], 0, 0
    while i < len(tokens):
        k = _kind(tokens[i].group(0), big)
        if k is None or k[0] in ("H", "B"):
            i += 1
            continue
        total, cur = 0, k[1]
        last = k[0]
        j = i + 1
        while j < len(tokens):
            nk = _kind(tokens[j].group(0), big)
            if nk is None:
                break
            if nk[0] == "H":
                if cur == 0:
                    break
                cur *= 100
                last = "unit"
            elif nk[0] == "B":
                total += (cur or 1) * nk[1]
                cur = 0
                last = "unit"
            elif nk[0] == "unit":
                if last == "digit":
                    break
                if cur and cur % 100 == 0 and nk[1] < 100:
                    cur += nk[1]
                elif cur == 0:
                    cur = nk[1]
                else:
                    break
            else:
                break
            j += 1
        value = total + cur
        k2 = j
        while k2 < len(tokens) and tokens[k2].group(0) in COUNTERS:
            k2 += 1
        parts.append(t[pos:tokens[i].start()])
        parts.append(_fmt(value))
        pos = tokens[k2 - 1].end()
        i = k2
    parts.append(t[pos:])
    return "".join(parts)


PUNCT_V2 = r'[।,\.\-‐-―\'"“”‘’()!?;:\s]'


def norm_v2(s):
    return re.sub(PUNCT_V2, "", canon_numbers(s, BIG))


def hunt_norm_v2(s):
    s = unicodedata.normalize("NFC", str(s)).replace("‌", "").replace("‍", "")
    return re.sub(PUNCT_V2, "", canon_numbers(s, BIG_HUNT))


PUNCT_SUB = r'[।,\.\-\'"“”‘’()!?;:\s]'


def norm_sub(s):
    return re.sub(PUNCT_SUB, "", str(s))


HUNT_PUNCT = re.compile(r'[।,\.\-‐-―\'"“”‘’()!?;:\s\[\]‌‍]+')


def hunt_norm(s):
    return HUNT_PUNCT.sub("", unicodedata.normalize("NFC", str(s)))


BN_RUN = re.compile(r"[ঀ-৿]+|\d+")


def bn_tok(s):
    return set(BN_RUN.findall(str(s)))


def jacc(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


# --- deterministic Bengali OCR repair (work/source_hunt/site_match.py) -------
# Legacy Bijoy/ASCII scans split o-kar / au-kar into an e-kar plus an aa-kar or
# au-length-mark, in either order. Every bank key and every row string goes
# through this before matching, otherwise scanned MCQ banks silently never hit.
_OCR_PAIRS = (
    ("াে", "ো"),   # aa-kar + e-kar  -> o-kar
    ("ো", "ো"),   # e-kar  + aa-kar -> o-kar
    ("ৗে", "ৌ"),   # au-mark + e-kar -> au-kar
    ("ৌ", "ৌ"),   # e-kar + au-mark -> au-kar
)


def ocr_fix(s):
    if not s:
        return s
    s = str(s)
    for a, b in _OCR_PAIRS:
        s = s.replace(a, b)
    return s




In [ ]:
# %% ------------------------------------------------------------------
# Cell 6b: math-word-problem detector (verbatim port of work/rules.py
#          is_math_prompt) — routes closed-book arithmetic rows to the symbolic
#          solver and then to the thinking-mode math router.
MATH_PAT = re.compile(
    r"(কত টাকা|শতকরা|লাভ|ক্ষতি|সুদ|আসল|গড়|অনুপাত|যোগফল|বিয়োগফল|গুণফল|ভাগফল"
    r"|কত অংশ|মোট কত|বয়সের|বয়স কত|গতিবেগ|কত সময়|কত দিন|কত ঘন্টা|সমষ্টি"
    r"|বিক্রয়মূল্য|ক্রয়মূল্য|আয়তন কত|ক্ষেত্রফল|পরিসীমা|শতাংশ)")


def is_math_prompt(prompt, context=""):
    if context:
        return False
    has_kw = bool(MATH_PAT.search(prompt))
    n_nums = len(re.findall(r"[\d০-৯]+", prompt))
    return has_kw and n_nums >= 1 or n_nums >= 3




In [ ]:
# %% ------------------------------------------------------------------
# Cell 6c: TIER MODULE BRIDGE (held-out mode only).
#
# Several final_entry tiers are large, self-contained, heavily audited Bengali-linguistic
# programs (work/somas_tier.py, biparit_sandhi_tier.py, idiom_upasarga_tier.py,
# math_solve.py, cb_last_tier.py, cb_wiki_tier.py, gram_match.py, ctx_bio_tier.py,
# wikt_idioms.py, idiom_tail4.py — ~2,500 lines in total). They are shipped
# VERBATIM as the attached Kaggle code dataset rather than re-transcribed here:
# re-typing them would risk silent divergence from the code that actually
# produced the 0.954 submission, which is the one thing Phase-2 must not do.
#
# This cell puts that code on sys.path, installs a `common` shim pointing at the
# MOUNTED competition data (the modules import `common.load_test`), neutralizes
# the one module that calls os.chdir() to a developer-machine path at import
# time, and imports each module defensively. A module that fails to import
# disables exactly its own tier — its rows fall through to the next layer.
# STAGING. The tier modules resolve their data files RELATIVE TO THEIR OWN
# DIRECTORY (idiom_upasarga_tier reads assets/*.json and wikt_pages.json at
# IMPORT time; cb_wiki_tier reads wiki_articles/<title>.txt; wikisource_tier
# reads source_data/wikisource_tier_cache.json; idiom_upasarga_tier.exam_pairs
# reads source_hunt/livemcq/qa2.json and source_hunt/sites/*/*.json). Kaggle
# mounts are read-only and the banks dataset was uploaded with FLATTENED names,
# so the code dataset alone is not a working module root. This cell rebuilds the
# expected layout under a WRITABLE staging dir, sourcing every file by basename
# from anywhere under /kaggle/input. When the mounted root already has the full
# layout (local dev against the repo's work/) it is used as-is and nothing is
# copied, so local reproduction is bit-identical to the shipped build.
import shutil

STAGE_DIR = os.path.join(OUT_DIR, "halluc_tiers")
TIER_CODE_DIR = None


def _code_root():
    """Directory containing the tier modules: the code dataset, else local work/."""
    for hit in sorted(glob.glob("/kaggle/input/**/somas_tier.py", recursive=True)):
        return os.path.dirname(hit)
    if os.path.isfile(os.path.join(_WORK, "somas_tier.py")):
        return _WORK
    return None


# path-under-the-module-root -> basenames to look for, flat spelling first
TIER_DATA_DEPS = [
    ("assets/bn_grammar_kb.json",   ["bn_grammar_kb.json"]),
    ("assets/bengali_idioms.json",  ["bengali_idioms.json"]),
    ("assets/harvested_gloss.json", ["harvested_gloss.json"]),
    ("wikt_pages.json",             ["wikt_pages.json"]),
    ("source_data/wikisource_tier_cache.json", ["wikisource_tier_cache.json"]),
    ("cb379_idx.json",              ["cb379_idx.json"]),
    ("source_hunt/livemcq/qa2.json", ["livemcq_qa2.json", "qa2.json"]),
]


def _layout_complete(root):
    """True iff `root` can serve as the module root with NO copying — i.e. every
    data dependency is already at the path its module expects. The tier-code
    dataset deliberately does NOT carry the question banks (they are a separate
    15 MB dataset), so on Kaggle this is always False and staging always runs;
    the local dev repo has the full tree and short-circuits."""
    return (all(os.path.exists(os.path.join(root, d)) for d, _ in TIER_DATA_DEPS)
            and bool(glob.glob(os.path.join(root, "wiki_articles", "*.txt")))
            and bool(glob.glob(os.path.join(root, "source_hunt", "sites",
                                            "*", "qa.json"))))


def stage_tier_code(root):
    """Copy the modules + every data dependency into STAGE_DIR. Returns the dir
    to import from. Never raises: a dependency that cannot be found is reported
    and disables exactly the tier that needs it."""
    if _layout_complete(root):
        print(f"tier code root: {root} (complete layout — no staging needed)")
        return root
    os.makedirs(STAGE_DIR, exist_ok=True)
    n_py = 0
    for p in sorted(glob.glob(os.path.join(root, "*.py"))):
        shutil.copy2(p, os.path.join(STAGE_DIR, os.path.basename(p)))
        n_py += 1
    got, absent = [], []
    for dest, names in TIER_DATA_DEPS:
        out = os.path.join(STAGE_DIR, dest)
        os.makedirs(os.path.dirname(out) or STAGE_DIR, exist_ok=True)
        src = None
        for nm in names:                       # dataset root, then anywhere
            src = (find_input(nm, os.path.join(root, dest), required=False)
                   or find_input(nm, os.path.join(_WORK, dest), required=False))
            if src:
                break
        if src:
            shutil.copy2(src, out)
            got.append(dest)
        else:
            absent.append(dest)
    # wiki_articles/<title>.txt — the cb_wiki corpus, shipped in the code dataset
    _wa = find_dir("wiki_articles", os.path.join(root, "wiki_articles")) \
        or find_dir("wiki_articles", os.path.join(_WORK, "wiki_articles"))
    _n_wa = 0
    if _wa:
        os.makedirs(os.path.join(STAGE_DIR, "wiki_articles"), exist_ok=True)
        for p in sorted(glob.glob(os.path.join(_wa, "*"))):
            shutil.copy2(p, os.path.join(STAGE_DIR, "wiki_articles",
                                         os.path.basename(p)))
            _n_wa += 1
    else:
        absent.append("wiki_articles/")
    # source_hunt/sites/<domain>/qa.json — flat `sites_<domain>_qa.json` on Kaggle
    _n_site = 0
    for p in find_inputs(SITE_BANK_PATTERNS, (root, _WORK)):
        b = os.path.basename(p)
        m = SITE_FLAT_RE.match(b)
        dom = m.group(1) if m else os.path.basename(os.path.dirname(p))
        d = os.path.join(STAGE_DIR, "source_hunt", "sites", dom)
        os.makedirs(d, exist_ok=True)
        shutil.copy2(p, os.path.join(d, "qa.json"))
        _n_site += 1
    print(f"tier code STAGED into {STAGE_DIR}: {n_py} modules, "
          f"{len(got)} data files, {_n_wa} wiki articles, {_n_site} site banks")
    if absent:
        print(f"  staging could NOT resolve: {absent} -> the tiers that need them "
              f"will decline to predict (their rows fall through)")
    return STAGE_DIR


if LIVE:
    _root = _code_root()
    if _root is None:
        print("!!! tier code dir NOT attached -> the module-backed deterministic "
              "tiers are DISABLED; their rows fall through to the LLM judges\n"
              "    attach mdmeheduzzaman/bengali-halluc-tier-code")
    else:
        try:
            TIER_CODE_DIR = stage_tier_code(_root)
        except Exception as _e:
            print(f"!!! tier code staging FAILED ({type(_e).__name__}: {_e}) -> "
                  f"falling back to the raw mount {_root}")
            TIER_CODE_DIR = _root

TIER_MODULES = {}
if TIER_CODE_DIR:
    import types as _types

    if TIER_CODE_DIR not in sys.path:
        sys.path.insert(0, TIER_CODE_DIR)

    # `common` shim: the tier modules call load_test()/load_samples(); point them
    # at the MOUNTED competition files rather than the developer's repo path.
    _shim = _types.ModuleType("common")
    _shim.load_test = lambda: [dict(r) for r in T]
    _shim.load_samples = lambda: [dict(r) for r in S]
    _shim.f1_halluc = f1_halluc
    _shim.toks = bn_tok
    # The shipped work/common.py also exports BN / NUM / f1_macro / report and
    # some tier modules do `from common import ... report` at module scope. A
    # missing name would raise ImportError and silently disable that tier, so
    # the shim mirrors the full public surface of work/common.py.
    _shim.BN = re.compile(r"[ঀ-৿]+")
    _shim.NUM = re.compile(r"[০-৯0-9]+")

    def _f1_macro(y_true, y_pred):
        def _f1(pos):
            tp = sum(1 for t, p in zip(y_true, y_pred) if t == pos and p == pos)
            fp = sum(1 for t, p in zip(y_true, y_pred) if t != pos and p == pos)
            fn = sum(1 for t, p in zip(y_true, y_pred) if t == pos and p != pos)
            pr = tp / (tp + fp) if tp + fp else 0.0
            rc = tp / (tp + fn) if tp + fn else 0.0
            return 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        return (_f1(0) + _f1(1)) / 2

    _shim.f1_macro = _f1_macro
    _shim.report = lambda y_true, y_pred, name="": print(
        f"{name:28s} F1(halluc)={f1_halluc(y_true, y_pred):.3f}  "
        f"macroF1={_f1_macro(y_true, y_pred):.3f}")
    _shim.DATA = os.path.dirname(test_path)
    sys.modules["common"] = _shim

    _real_chdir = os.chdir
    os.chdir = lambda p: None            # neutralize import-time chdir

    for _name in ["somas_tier", "biparit_sandhi_tier", "idiom_upasarga_tier",
                  "math_solve", "cb_last_tier", "cb_wiki_tier", "gram_match",
                  "ctx_bio_tier", "wikt_idioms", "idiom_tail", "idiom_tail2",
                  "idiom_tail4", "wikisource_tier", "bn_num"]:
        try:
            TIER_MODULES[_name] = __import__(_name)
        except Exception as _e:
            print(f"  tier module {_name}: UNAVAILABLE ({type(_e).__name__}: {_e})")
        # work/cb_last_tier.py does `sys.path.insert(0, "<dev path>/work")` at
        # import time. That path does not exist on Kaggle (harmless), but in a
        # LOCAL rehearsal it would shadow the staged root and later modules would
        # load from the repo instead. Re-assert the staged root after every
        # import so every tier is provably the shipped copy under TIER_CODE_DIR.
        if sys.path[0] != TIER_CODE_DIR:
            sys.path.insert(0, TIER_CODE_DIR)
    os.chdir = _real_chdir
    _from_stage = sum(1 for _m in TIER_MODULES.values()
                      if os.path.dirname(os.path.abspath(getattr(_m, "__file__", "")))
                      == os.path.abspath(TIER_CODE_DIR))
    print(f"  tier modules resolved from the staged root: "
          f"{_from_stage}/{len(TIER_MODULES)}")

    # OFFLINE ENFORCEMENT. work/wikisource_tier.py was written as an online
    # harvester: on a cache miss it POSTs to accessibledictionary.gov.bd. Phase-2
    # forbids network access at inference, so its HTTP session is replaced with a
    # stub that fails instantly. The tier then resolves ONLY from the attached
    # cache snapshot — fresh held-out heads simply miss and fall through.
    _ws = TIER_MODULES.get("wikisource_tier")
    if _ws is not None:
        class _NoNetSession:
            def post(self, *a, **k):
                raise _ws.requests.RequestException("offline: network disabled")

            def get(self, *a, **k):
                raise _ws.requests.RequestException("offline: network disabled")

        _ws.S = _NoNetSession()
        # Kill the module's rate-limit back-off (time.sleep(0.25) after every
        # dead request — real seconds once a held-out fold misses the cache
        # thousands of times). Rebind the module's OWN reference to a shim
        # rather than assigning `_ws.time.sleep`: `_ws.time` IS the stdlib time
        # module, so that spelling made time.sleep a no-op process-wide for
        # every library in the kernel.
        _ws.time = _types.SimpleNamespace(sleep=lambda *a, **k: None,
                                          time=time.time)
        _ws.save_cache = lambda *a, **k: None      # never write to a read-only mount
        print(f"  wikisource_tier: network disabled, "
              f"{len(getattr(_ws, 'cache', {}))} cached queries available")
    print(f"tier modules loaded: {sorted(TIER_MODULES)}")


def tier_call(module, fn, *a, **kw):
    """Call a tier module's pure function; return None on absence or error.
    A tier can never crash the run — it can only decline to predict."""
    m = TIER_MODULES.get(module)
    if m is None:
        return None
    f = getattr(m, fn, None)
    if f is None:
        return None
    try:
        return f(*a, **kw)
    except Exception:
        return None




In [ ]:
# %% ------------------------------------------------------------------
# Cell 7: HELD-OUT layer — leak (exact prompt+response in the labeled split)
leak_pred = {}
if LIVE:
    by_prompt = collections.defaultdict(list)
    for r in S:
        by_prompt[r["prompt_bn"].strip()].append(r)
    for i, r in enumerate(T):
        exact = [o for o in by_prompt.get(r["prompt_bn"].strip(), [])
                 if str(o["response_bn"]).strip() == str(r["response_bn"]).strip()]
        if exact:
            leak_pred[i] = int(exact[0]["label"])
    print(f"leak layer: {len(leak_pred)} test rows")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 8: HELD-OUT layer — ctx_gold: context rows matched back to their public
#         QA source datasets (port of work/source_hunt_ctx.py).
#         squad_bn's validation/test splits ARE TyDiQA-GoldP-bn; BanglaRQA /
#         IndicQA-bn / raw TyDiQA-GoldP are optional extra coverage.
ctx_gold, ctx_sub = {}, {}
entries = []


def load_ctx_sources():
    out = []

    def add_squad_data(d, source):
        for art in d["data"]:
            for para in art["paragraphs"]:
                ctx = para["context"]
                for qa in para["qas"]:
                    answers = list(dict.fromkeys(
                        a["text"] for a in qa.get("answers", []) if a.get("text")))
                    out.append(dict(ctx=ctx, question=qa["question"], answers=answers,
                                    answerable=bool(answers), source=source))

    # NAME FLATTENING — the same hazard the source banks document above. Kaggle's
    # dataset uploader DROPS SUBDIRECTORIES, so a published `squad_bn/<split>.json`
    # arrives at the mount root as a BARE `<split>.json`. The nested-only glob
    # this loop used to run therefore never matched on Kaggle, and ctx_gold — the
    # single highest-coverage context tier — silently produced ZERO rows.
    # Measured on a synthetic 5,000-row held-out fold: ctx_gold 0/2,709 ctx rows
    # and deterministic coverage 49.1%, against 1,835/2,709 and 86.0% once the
    # file is actually found. Every candidate is SHAPE-CHECKED before use (SQuAD
    # JSON is {"data": [{"paragraphs": [...]}, ...]}), so a generic basename like
    # `test.json` belonging to some unrelated attached dataset can never be
    # mistaken for a source corpus — a miss degrades to "tier disabled", never to
    # "tier grounded against garbage".
    def _squad_or_none(path):
        try:
            with open(path, encoding="utf-8") as f:
                d = json.load(f)
        except Exception:
            return None
        arts = d.get("data") if isinstance(d, dict) else None
        if not isinstance(arts, list) or not arts:
            return None
        a0 = arts[0]
        if not (isinstance(a0, dict) and isinstance(a0.get("paragraphs"), list)):
            return None
        return d

    def add_squad_json(path, source):
        """Parse + shape-check a SQuAD-format file, then add it. A file that is
        not SQuAD-shaped is skipped rather than raising — the other corpora and
        the rest of the pipeline must not die on one bad attachment."""
        d = _squad_or_none(path)
        if d is None:
            print(f"  (skipping {path}: not SQuAD-shaped)")
            return False
        add_squad_data(d, source)
        return True

    _squad_roots = (os.path.join(_WORK, "source_data", "data"), _WORK)
    found = False
    for split in ["train", "validation", "test"]:
        # priority order: nested (local dev / an unflattened mount), the
        # recommended flat upload name, then the bare flattened basename.
        for _pat in (f"squad_bn/{split}.json", f"squad_bn_{split}.json",
                     f"{split}.json"):
            _hit = None
            for _p in find_inputs([_pat], _squad_roots):
                _d = _squad_or_none(_p)
                if _d is not None:
                    _hit = (_p, _d)
                    break
            if _hit:
                add_squad_data(_hit[1], f"squad_bn/{split}")
                found = True
                print(f"  squad_bn/{split}: {_hit[0]}")
                break
    if not found:
        print("!!! squad_bn NOT attached -> ctx_gold DISABLED; every ctx row routes "
              "to the deterministic tiers / judge / substring rule\n"
              "    (attach the squad_bn corpus; flat basenames are searched too)")

    for name in ["tydiqa-goldp-v1.1-train.json", "tydiqa-goldp-v1.1-dev.json"]:
        p = find_input(name, required=False)
        if p:
            d = json.load(open(p, encoding="utf-8"))["data"]
            for art in d:
                for para in art["paragraphs"]:
                    for qa in para["qas"]:
                        if not str(qa.get("id", "")).startswith("bengali"):
                            continue
                        answers = list(dict.fromkeys(
                            a["text"] for a in qa.get("answers", []) if a.get("text")))
                        out.append(dict(ctx=para["context"], question=qa["question"],
                                        answers=answers, answerable=bool(answers),
                                        source="tydiqa-goldp-bn"))

    p = find_input("indicqa.bn.json",
                   os.path.join(_WORK, "source_data", "indicqa", "data",
                                "indicqa.bn.json"), required=False)
    if p:
        add_squad_json(p, "indicqa_bn")

    for split in ["Train", "Validation", "Test"]:
        p = find_input(f"BanglaRQA/{split}.json",
                       os.path.join(_WORK, "source_data", "BanglaRQA", f"{split}.json"),
                       required=False)
        if not p:
            continue
        for e in json.load(open(p, encoding="utf-8"))["data"]:
            for qa in e["qas"]:
                ans = qa.get("answers", {})
                answers = list(dict.fromkeys(
                    a.strip() for a in ans.get("answer_text", []) if a and a.strip()))
                out.append(dict(ctx=e["context"], question=qa["question_text"],
                                answers=answers,
                                answerable=qa.get("is_answerable") in ("1", 1, True),
                                source=f"BanglaRQA/{split}"))
    return out


CTX_SRC_OK = False
if LIVE:
    t_ctx = time.time()
    entries = load_ctx_sources()
    print(f"ctx source entries: {len(entries)} "
          f"{dict(collections.Counter(e['source'].split('/')[0] for e in entries))}")
    from sklearn.feature_extraction.text import TfidfVectorizer
    CTX_SRC_OK = len(entries) > 0

if CTX_SRC_OK:
    qidx = {}
    for i, e in enumerate(entries):
        qidx.setdefault(hunt_norm(e["question"]), []).append(i)
    ctxs = list(dict.fromkeys(e["ctx"] for e in entries))
    nctx = [hunt_norm(c) for c in ctxs]
    ctx2i = {c: i for i, c in enumerate(ctxs)}
    pidx = {}
    for i, nc in enumerate(nctx):
        pidx.setdefault(nc[:80], []).append(i)
    ent_by_ctx = {}
    for i, e in enumerate(entries):
        ent_by_ctx.setdefault(ctx2i[e["ctx"]], []).append(i)
    ctx_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=200000)
    ctx_X = ctx_vec.fit_transform(nctx)
    print(f"ctx source index built ({time.time()-t_ctx:.0f}s)")
else:
    qidx, ctxs, nctx, ctx2i, pidx, ent_by_ctx = {}, [], [], {}, {}, {}
    ctx_vec, ctx_X = None, None


def match_row(row_ctx, row_prompt):
    """(entry_idx, match_type) or None. Tiers: exact question (+ctx check),
    context prefix/substring + fuzzy question, char-TF-IDF nearest context."""
    if not CTX_SRC_OK:
        return None
    nq, nc = hunt_norm(row_prompt), hunt_norm(row_ctx)
    cands = qidx.get(nq, [])
    if cands:
        best, bt = None, -1
        for i in cands:
            enc = nctx[ctx2i[entries[i]["ctx"]]]
            if nc and (nc in enc or enc in nc):
                return i, "q_exact+ctx_sub"
            t = jacc(bn_tok(row_ctx), bn_tok(entries[i]["ctx"]))
            if t > bt:
                bt, best = t, i
        if bt >= 0.25:
            return best, "q_exact+ctx_jacc"
        return best, "q_exact_only"
    ctx_hits = pidx.get(nc[:80], [])
    if not ctx_hits:
        key = nc[:40]
        if len(key) >= 30:
            ctx_hits = [i for i, c in enumerate(nctx) if key in c]
    if ctx_hits:
        qt = bn_tok(row_prompt)
        best, bt = None, -1
        for ci in ctx_hits:
            for i in ent_by_ctx.get(ci, []):
                t = jacc(qt, bn_tok(entries[i]["question"]))
                if t > bt:
                    bt, best = t, i
        if bt >= 0.55:
            return best, "ctx_match+q_jacc"
    q = ctx_vec.transform([nc])
    sims = (ctx_X @ q.T).toarray().ravel()
    t = int(np.argmax(sims))
    if sims[t] < 0.90:
        return None
    qt = bn_tok(row_prompt)
    best, bt = None, -1.0
    for ei in ent_by_ctx.get(t, []):
        tj = jacc(qt, bn_tok(entries[ei]["question"]))
        if tj > bt:
            bt, best = tj, ei
    if best is not None and bt >= 0.70:
        return best, "tfidf_ctx"
    return None


def _mask_prompt_echo(nr, np_, minlen=8):
    """Blank spans of the normalized response (>= minlen chars) that also occur
    in the normalized prompt — a gold hit inside a question echo proves nothing."""
    masked = list(nr)
    L, i = len(nr), 0
    while i < L:
        j = i + minlen
        if j <= L and nr[i:j] in np_:
            while j <= L and nr[i:j] in np_:
                j += 1
            span = j - 1 - i
            for k in range(i, i + span):
                masked[k] = "\x00"
            i += span
        else:
            i += 1
    return "".join(masked)


def gold_rule_v2(response, prompt, answers):
    """Faithful iff a gold answer appears in the response OUTSIDE prompt-echo
    text (or the response is, or is part of, the gold)."""
    nr = hunt_norm_v2(response)
    np_ = hunt_norm_v2(prompt)
    nr_masked = _mask_prompt_echo(nr, np_)
    for g in answers:
        ng = hunt_norm_v2(g)
        if not ng:
            continue
        if ng in nr_masked or (nr and nr in ng):
            return 1
        if ng in nr and len(nr) <= len(ng) + 6:
            return 1
    return 0


YEARQ = re.compile(r"কত সালে|কোন সালে|কত খ্রিস্টাব্দে|কোন বছর|কবে |কবে\?|তারিখ")


def suspect_gold(prompt, answers):
    """Year/date question whose gold carries no number -> wrong-typed source gold
    (TyDiQA annotation noise); do not trust it for labeling."""
    if not YEARQ.search(prompt):
        return False
    for g in answers:
        if re.search(r"[\d০-৯]", str(g)) or "#" in hunt_norm_v2(g):
            return False
    return True


def ctx_gold_pred(r):
    m = match_row(r["context"], r["prompt_bn"])
    if not m:
        return None
    e = entries[m[0]]
    if not e["answers"] or suspect_gold(r["prompt_bn"], e["answers"]):
        return None
    return gold_rule_v2(r["response_bn"], r["prompt_bn"], e["answers"])


if LIVE and CTX_SRC_OK:
    _val = [(ctx_gold_pred(S[i]), S[i]["label"]) for i in CTX_S]
    _cov = [(p, y) for p, y in _val if p is not None]
    if _cov:
        print(f"ctx_gold sample validation: matched {len(_cov)}/{len(CTX_S)} labeled "
              f"ctx rows, gold-rule accuracy = "
              f"{sum(p == y for p, y in _cov)/len(_cov):.3f}")

if LIVE:
    _t0 = time.time()
    for i in CTX_T:
        if i in leak_pred:
            continue
        p = ctx_gold_pred(T[i])
        if p is not None:
            ctx_gold[i] = p
    print(f"ctx_gold: {len(ctx_gold)}/{len(CTX_T)} test ctx rows ({time.time()-_t0:.0f}s)")

    # ctx_bio: Bengali ordinal-suffix date normalization rescue. A ctx_gold row
    # scored 0 flips to 1 iff the response equals a gold answer after date
    # normalization (work/ctx_bio_tier.py). Zero parameters.
    _bio_n = 0
    _norm_date = TIER_MODULES.get("ctx_bio_tier")
    if _norm_date is not None and CTX_SRC_OK:
        for i, p in list(ctx_gold.items()):
            if p != 0:
                continue
            m = match_row(T[i]["context"], T[i]["prompt_bn"])
            if not m:
                continue
            nd = tier_call("ctx_bio_tier", "norm_date", T[i]["response_bn"])
            if nd is None:
                continue
            for g in entries[m[0]]["answers"]:
                if nd and nd == tier_call("ctx_bio_tier", "norm_date", g):
                    ctx_gold[i] = 1
                    _bio_n += 1
                    break
    print(f"ctx_bio (ordinal-date rescue): {_bio_n} ctx_gold rows flipped 0 -> 1")

    # Parameter-free verbatim-substring rule: the T2_substring rung of the
    # governor ladder AND the universal fallback for any undecided ctx row.
    for i in CTX_T:
        r = T[i]
        nr, nc = norm_sub(r["response_bn"]), norm_sub(r["context"])
        nr2, nc2 = norm_v2(r["response_bn"]), norm_v2(r["context"])
        ctx_sub[i] = 1 if ((nr and nr in nc) or (nr2 and nr2 in nc2)) else 0
    _un = [i for i in CTX_S if ctx_gold_pred(S[i]) is None] if CTX_SRC_OK else list(CTX_S)
    if _un:
        _yp = []
        for i in _un:
            r = S[i]
            nr, nc = norm_sub(r["response_bn"]), norm_sub(r["context"])
            nr2, nc2 = norm_v2(r["response_bn"]), norm_v2(r["context"])
            _yp.append(1 if ((nr and nr in nc) or (nr2 and nr2 in nc2)) else 0)
        print(f"substring rule sample validation (unmatched ctx rows, n={len(_un)}): "
              f"F1(halluc) = {f1_halluc([S[i]['label'] for i in _un], _yp):.3f}")
    emit_submission(tag="(after ctx_gold + substring rule)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 9: HELD-OUT layer — ctx deterministic Bengali-grammar tiers.
# সমাস / সন্ধি / বিপরীত শব্দ / উপসর্গ / বাগধারা. These authored-context rows do
# NOT state their answer in the context, so a passage-faithfulness judge has
# nothing to check and answers "faithful" on ~all of them — it is structurally
# mismatched. Standard NCTB grammar canon decides them exactly.
ctx_det, ctx_last_bip, ctx_last_gloss = {}, {}, {}
if LIVE:
    _pool = [i for i in CTX_T if i not in leak_pred and i not in ctx_gold]
    for i in _pool:
        ctx_i, q, resp = T[i]["context"], T[i]["prompt_bn"], T[i]["response_bn"]
        p = None
        # tier order matches the shipped router's `det` setdefault precedence:
        # grammar (সমাস) -> সন্ধি/বিপরীত -> উপসর্গ/বাগধারা
        for mod, fn, args in (
            ("somas_tier", "judge", (ctx_i, resp)),
            ("somas_tier", "sandhi_judge", (ctx_i, resp)),
            ("biparit_sandhi_tier", "sandhi_judge", (ctx_i, resp)),
            ("biparit_sandhi_tier", "biparit_judge", (ctx_i, resp)),
            ("idiom_upasarga_tier", "judge_upasarga", (ctx_i, q, resp)),
            ("idiom_upasarga_tier", "judge_idiom", (q, resp)),
        ):
            v = tier_call(mod, fn, *args)
            if isinstance(v, (list, tuple)):        # some return (pred, how)
                v = v[0] if v else None
            if v in (0, 1):
                p = int(v)
                break
        if p is not None:
            ctx_det[i] = p
    print(f"ctx deterministic grammar tiers: {len(ctx_det)} rows "
          f"(of {len(_pool)} unmatched ctx rows)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 10: HELD-OUT layer — closed-book source matching.
#          (a) hishab/bangla-mmlu gold match, with the OCR normalizer;
#          (b) livemcq + sibling exam-site QA banks;
#          (c) the bn_grammar_kb canon tier.
# All three share one matcher core (port of work/source_hunt/matcher.py, which
# is also the base of work/source_hunt/cb2_tiers.py).
import pandas as pd


def m_norm(s):
    if s is None:
        return ""
    s = unicodedata.normalize("NFC", ocr_fix(str(s)))
    s = s.replace("‌", "").replace("‍", "").replace("﻿", "")
    s = re.sub(r'[“”"‘’\'`´]', "", s)
    s = re.sub(r'[?？।.,;:!()\[\]{}<>—–\-_/\\|*#@%&+=~^]', " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()


def m_norma(s):
    return m_norm(s).translate(BN2ASCII)


M_STOP = set("একটি একজন হলো হল হচ্ছে থেকে এর এবং ও সালে সাল খ্রিস্টাব্দে "
             "খ্রিস্টাব্দ খৃষ্টাব্দ খ্রি খৃঃ খ্রিঃ খ্রী সন সনে".split())


def m_toks(s):
    return [t for t in m_norma(s).split() if t not in M_STOP]


def m_nums(s):
    return re.findall(r"\d+\.?\d*", m_norma(s))


M_BN_RE = re.compile(r"[ঀ-৿]")
M_ASCII_RE = re.compile(r"[a-z]")


def script_of(s):
    s = m_norma(s)
    bn, en = bool(M_BN_RE.search(s)), bool(M_ASCII_RE.search(s))
    if bn and not en:
        return "bn"
    if en and not bn:
        return "en"
    return "mixed"


def ans_match(resp, gold):
    """True / False / None(abstain): does the response state the gold answer?"""
    r, g = m_norma(resp), m_norma(gold)
    if not g or not r:
        return None
    if r == g or g in r or r in g:
        return True
    sr, sg = script_of(r), script_of(g)
    if {sr, sg} == {"bn", "en"}:
        return None                                   # cross-script: abstain
    rt, gt = set(m_toks(resp)), set(m_toks(gold))
    if gt and rt:
        if len(rt & gt) / len(rt | gt) >= 0.75:
            return True

        def tok_eq(a, b):
            if a == b:
                return True
            if a.isdigit() or b.isdigit():
                return False
            return _ratio(a, b) >= 80

        cov_g = sum(1 for g_ in gt if any(tok_eq(g_, r_) for r_ in rt)) / len(gt)
        cov_r = sum(1 for r_ in rt if any(tok_eq(r_, g_) for g_ in gt)) / len(rt)
        if cov_g == 1.0 and cov_r >= 0.6:
            return True
    gn, rn = m_nums(gold), m_nums(resp)
    g_nonnum = [t for t in gt if not re.fullmatch(r"\d+\.?\d*%?", t)]
    if gn and rn and not g_nonnum:
        return set(gn) == set(rn)
    return False


_LETTER_RE = re.compile(r"[a-zঀ-৿]")


def ans_match_strict(resp, gold):
    """ans_match plus a stricter numeric rule for the expansion tiers: a numeric
    or symbolic gold demands digit-set equality; a subset abstains."""
    g = m_norma(gold)
    if g and not _LETTER_RE.search(g):
        gn, rn = set(m_nums(gold)), set(m_nums(resp))
        if not gn:
            return None
        if rn == gn:
            return True
        if gn <= rn:
            return None
        return False
    return ans_match(resp, gold)


BN_LETTERS = ["ক", "খ", "গ", "ঘ", "ঙ", "চ"]


def split_options(prompt):
    """If the prompt embeds options 'ক) ... খ) ...' return (stem, [opts])."""
    p = str(prompt)
    marks = list(re.finditer(r"([কখগঘঙচ])\s*[).:]", p))
    if len(marks) < 3 or marks[0].group(1) != "ক":
        return p, None
    stem = p[:marks[0].start()].strip(" ,;:-—?")
    opts = []
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(p)
        opts.append(p[m.end():end].strip(" ,;।?."))
    return stem, opts


def resolve_letter_response(resp, opts):
    r = str(resp).strip()
    m = re.match(r"^[\s(]*([কখগঘঙচ])[\s).।:]*$", r)
    if m and opts:
        i = BN_LETTERS.index(m.group(1))
        if i < len(opts):
            return opts[i]
    m2 = re.match(r"^[\s(]*([কখগঘঙচ])\s*[).।:]\s*(.+)$", r)
    if m2 and opts:
        i = BN_LETTERS.index(m2.group(1))
        if i < len(opts):
            return opts[i] + " " + m2.group(2)
    return None


def verdict(prompt, resp, golds, how, strict=False):
    """Vote gold variants -> (pred 0/1/None, how). Conflict/abstain semantics
    exactly as work/source_hunt/matcher.py predict_row."""
    am = ans_match_strict if strict else ans_match
    stem, opts = split_options(prompt)
    resp_eff = resp
    if opts is not None:
        lt = resolve_letter_response(resp, opts)
        if lt is not None:
            resp_eff = lt
    gnorms = set(m_norma(g) for g, _ in golds if m_norma(g))
    if not gnorms:
        return None, how + "-emptygold"
    votes = [am(resp_eff, g) for g, _ in golds if m_norma(g)]
    if any(v is True for v in votes):
        return 1, how
    if all(v is None for v in votes):
        return None, how + "-abstain"
    if len(gnorms) > 1:
        if not re.search(r"বানান|শুদ্ধ|অশুদ্ধ", str(prompt)):
            gl = [g for g, _ in golds if m_norma(g)]
            if all(ans_match(a, b) is True or ans_match(b, a) is True
                   for i, a in enumerate(gl) for b in gl[i + 1:]):
                return 0, how + "-agreedvariants"
        return None, how + "-conflict"
    return 0, how


def _fuzzy_q_guard(q, m):
    """Accept a fuzzy question match only if the differing tokens are near-
    identical spelling variants (no content-word or digit differences)."""
    if m_nums(q) != m_nums(m):
        return False
    qs, ms = set(m_toks(q)), set(m_toks(m))
    qd, md = list(qs - ms), list(ms - qs)
    for t in qd:
        if not any((not t.isdigit() and not u.isdigit() and _ratio(t, u) >= 85)
                   for u in md):
            if len(t) >= 3 or t.isdigit():
                return False
    for u in md:
        if not any((not u.isdigit() and not t.isdigit() and _ratio(t, u) >= 85)
                   for t in qd):
            if len(u) >= 3 or u.isdigit():
                return False
    return True


def bank_predict(prompt, resp, src, keys, fuzzy_thresh=BANK_FUZZY_THRESH):
    """Question-bank lookup: exact key, embedded-option stem key, then guarded
    fuzzy. `src` maps m_norm(question) -> [(gold, opts), ...]."""
    stem, opts = split_options(prompt)
    keyhows = [(m_norm(prompt), "exact")]
    if opts is not None:
        keyhows.append((m_norm(stem), "stem"))
    for k, h in keyhows:
        if k in src:
            return verdict(prompt, resp, src[k], h)
    if HAVE_RAPIDFUZZ and keys:
        for k, h in keyhows:
            if len(k) < 20:
                continue
            got = _rf_process.extractOne(k, keys, scorer=_rf_fuzz.ratio,
                                         score_cutoff=fuzzy_thresh)
            if got and _fuzzy_q_guard(k, got[0]):
                return verdict(prompt, resp, src[got[0]], h + "-fuzzy")
    return None, "nomatch"


# ---- (a) hishab/bangla-mmlu -------------------------------------------------
mmlu_src, mmlu_keys = {}, []
if LIVE:
    t_mmlu = time.time()
    # NAME FLATTENING, again: the parquet shards are named
    # `<split>-00000-of-00001.parquet` and live under a `hishab__bangla-mmlu/`
    # directory. Kaggle's uploader drops that directory, so the substring test
    # `"mmlu" in path` only survived if the DATASET SLUG happened to contain
    # "mmlu" — otherwise cb_gold(mmlu) silently scored 0 rows. Now: prefer paths
    # that do mention mmlu, but fall back to every attached parquet and select by
    # SCHEMA (question / choices / answer). Schema selection is what makes the
    # broad fallback safe — an unrelated parquet cannot be read as gold answers.
    _mmlu_roots = (os.path.join(_WORK, "source_hunt", "hishab__bangla-mmlu", "data"),
                   _WORK)
    _all_pq = find_inputs(["*.parquet"], _mmlu_roots)
    _named = [p for p in _all_pq if "mmlu" in os.path.basename(p).lower()
              or "mmlu" in p.lower()]

    def _has_mmlu_schema(path):
        cols = None
        try:                                   # cheap: schema only, no row read
            import pyarrow.parquet as _pq
            cols = set(_pq.read_schema(path).names)
        except Exception:
            try:
                cols = set(pd.read_parquet(path).columns)
            except Exception:
                return False
        return {"question", "choices", "answer"} <= cols

    mmlu_files = [p for p in (_named or _all_pq) if _has_mmlu_schema(p)]
    if not mmlu_files:
        print("!!! bangla-mmlu NOT attached -> the cb-gold mmlu tiers are DISABLED; "
              "those rows fall through to the banks / judges"
              + (f" ({len(_all_pq)} parquet file(s) attached, none with the "
                 f"question/choices/answer schema)" if _all_pq else ""))
    else:
        print(f"bangla-mmlu shards: {len(mmlu_files)} "
              f"({', '.join(os.path.basename(p) for p in mmlu_files[:4])}"
              f"{' ...' if len(mmlu_files) > 4 else ''})")
        mmlu = pd.concat([pd.read_parquet(p) for p in sorted(mmlu_files)])
        for _, row in mmlu.iterrows():
            q = m_norm(row["question"])                 # m_norm applies ocr_fix
            ch = [str(c) for c in row["choices"]]
            al = row["answer"]
            idx = ord(al) - ord("A") if isinstance(al, str) and len(al) == 1 else None
            gold = ch[idx] if idx is not None and 0 <= idx < len(ch) else None
            if q and gold is not None:
                mmlu_src.setdefault(q, []).append((gold, ch))
        print(f"bangla-mmlu: {len(mmlu)} rows -> {len(mmlu_src)} unique questions "
              f"({time.time()-t_mmlu:.0f}s)")
    mmlu_keys = list(mmlu_src.keys())

# expansion-tier canonical keys (Bengali orthographic variants; not fitted)
PHON_MAP = {
    "ী": "ি", "ূ": "ু", "ঈ": "ই", "ঊ": "উ", "ণ": "ন", "শ": "স", "ষ": "স",
    "য": "জ", "ঙ": "ং", "ঞ": "ন", "খ": "ক", "ঘ": "গ", "ছ": "চ", "ঝ": "জ",
    "ঠ": "ট", "ঢ": "ড", "থ": "ত", "ধ": "দ", "ভ": "ব", "ফ": "প",
    "ৎ": "ত", "ঃ": "", "্": "", "ঁ": "", "ৌ": "ো", "ৈ": "ে",
}
_PHON_TR = {ord(k): v for k, v in PHON_MAP.items()}
_NUKTA_FIX = [("য়", "জ"), ("ড়", "র"), ("ঢ়", "র"), ("়", "")]


def phon(s):
    for a, b in _NUKTA_FIX:
        s = s.replace(a, b)
    return s.translate(_PHON_TR)


BOILER = {"নিচের", "নীচের", "নিম্নের", "নিম্নে", "নিচে", "উল্লিখিত", "প্রদত্ত",
          "প্রশ্ন", "উদ্দীপকের"}


def k_nospace(s):
    return m_norma(s).replace(" ", "")


def k_phon(s):
    return phon(k_nospace(s))


def k_bag(s):
    ts = [t for t in (phon(t) for t in m_norma(s).split() if t not in BOILER) if t]
    return " ".join(sorted(ts)) if len(ts) >= 3 else None


KEYFNS = [("tier1a", k_nospace), ("tier1b", k_phon), ("tier1c", k_bag)]
alt_idx = {name: {} for name, _ in KEYFNS}
for q in mmlu_src:
    for name, fn in KEYFNS:
        k = fn(q)
        if k and len(k) >= 12:
            alt_idx[name].setdefault(k, []).append(q)


def tier1_match(prompt):
    stem, opts = split_options(prompt)
    cands = [prompt] + ([stem] if opts is not None else [])
    for name, fn in KEYFNS:
        for c in cands:
            k = fn(c)
            if not k or len(k) < 12:
                continue
            qkeys = alt_idx[name].get(k)
            if qkeys:
                golds = []
                for q in qkeys:
                    golds.extend(mmlu_src[q])
                return name, golds
    return None


_VOWEL_SIGNS = set("ািীুূৃেৈোৌ")


def _pair_ok(a, b):
    if a.isdigit() or b.isdigit():
        return False
    pa, pb = phon(a), phon(b)
    if pa == pb:
        return True
    if not pa or not pb or pa[0] != pb[0]:
        return False
    return _ratio(pa, pb) >= 80


def _token_pairing_ok(qd, md):
    if len(qd) > 2 or len(md) > 2:
        return False
    used = set()
    for t in qd:
        hit = None
        for u in md:
            if u not in used and _pair_ok(t, u):
                hit = u
                break
        if hit is None:
            if len(t) >= 3 or t.isdigit():
                return False
        else:
            used.add(hit)
    for u in md:
        if u in used:
            continue
        if not any(_pair_ok(t, u) for t in qd):
            if len(u) >= 3 or u.isdigit():
                return False
    return True


def _vowel_sign_diff_ok(q, m):
    a, b = phon(q.replace(" ", "")), phon(m.replace(" ", ""))
    if a == b:
        return True
    changed = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
        if tag == "equal":
            continue
        if not all(c in _VOWEL_SIGNS for c in a[i1:i2] + b[j1:j2]):
            return False
        changed += max(i2 - i1, j2 - j1)
    return changed <= 3


def strict_guard(q, m):
    if m_nums(q) != m_nums(m):
        return False
    lq, lm = len(q), len(m)
    if min(lq, lm) / max(lq, lm) <= 0.9:
        return False
    qs, ms = set(m_toks(q)), set(m_toks(m))
    if _token_pairing_ok(sorted(qs - ms), sorted(ms - qs)):
        return True
    return _vowel_sign_diff_ok(q, m)


def cb_gold_pred(prompt, resp):
    """(pred 0/1 or None, how). tier0 exact/stem + guarded fuzzy@93, then the
    tier1a/b/c canonical keys and tier2 strict fuzzy@90 (both strict)."""
    if not mmlu_src:
        return None, "nomatch"
    stem, opts = split_options(prompt)
    keys = [(m_norm(prompt), "exact")]
    if opts is not None:
        keys.append((m_norm(stem), "stem"))
    for k, h in keys:
        if k in mmlu_src:
            return verdict(prompt, resp, mmlu_src[k], h)
    if HAVE_RAPIDFUZZ:
        for k, h in keys:
            if len(k) < 20:
                continue
            got = _rf_process.extractOne(k, mmlu_keys, scorer=_rf_fuzz.ratio,
                                         score_cutoff=93.0)
            if got and _fuzzy_q_guard(k, got[0]):
                return verdict(prompt, resp, mmlu_src[got[0]], h + "-fuzzy")
    t1 = tier1_match(prompt)
    if t1 is not None:
        name, golds = t1
        return verdict(prompt, resp, golds, name, strict=True)
    if HAVE_RAPIDFUZZ:
        for k, h in keys:
            if len(k) < 20:
                continue
            for mkey, score, _ in _rf_process.extract(
                    k, mmlu_keys, scorer=_rf_fuzz.ratio, score_cutoff=90.0, limit=5):
                if strict_guard(k, mkey):
                    return verdict(prompt, resp, mmlu_src[mkey],
                                   f"tier2({score:.0f})", strict=True)
    return None, "nomatch"


# ---- (b) livemcq + sibling-site QA banks ------------------------------------
# Each bank is a JSON list of {q, ans, opts, src}. Bank keys are built with the
# SAME m_norm (hence the same OCR repair) as the row keys, and open-ended
# questions are dropped: their free-text golds are not comparable.
LET_PREFIX = re.compile(r'^\s*[(]?([কখগঘঙচ])\s*[).।:৷]\s*')
OPENQ = re.compile(r"পার্থক্য|কাকে বলে|কী বুঝ|কি বুঝ|ব্যাখ্যা কর|বর্ণনা কর"
                   r"|আলোচনা কর|লিখুন|লেখ|উদাহরণ দাও|সংজ্ঞা|অনুবাদ কর"
                   r"|ভাবসম্প্রসারণ|রচনা")


def build_bank(paths, drop_open=True):
    src, n_pairs, dropped = collections.defaultdict(list), 0, 0
    for p in paths:
        try:
            qa = json.load(open(p, encoding="utf-8"))
        except Exception as e:
            print(f"  bank {p}: unreadable ({e})")
            continue
        for x in qa:
            n_pairs += 1
            g = ocr_fix(LET_PREFIX.sub("", str(x.get("ans", ""))).strip())
            q = ocr_fix(str(x.get("q", "")))
            if not g or (drop_open and (len(g) > 70 or OPENQ.search(q))):
                dropped += 1
                continue
            src[m_norm(q)].append((g, x.get("opts", [])))
    return dict(src), n_pairs, dropped


livemcq_src, livemcq_keys = {}, []
sites_src, sites_keys = {}, []
if LIVE:
    # livemcq: flat `livemcq_qa2.json` (Kaggle) or nested `livemcq/qa2.json`
    # (local dev). LIVEMCQ_BANK_NAMES is ordered flat-first, qa2 before qa, and
    # the first hit wins — qa2.json supersedes qa.json.
    _lm_paths = []
    for _n in LIVEMCQ_BANK_NAMES:
        _p = find_input(_n, os.path.join(_WORK, "source_hunt", "livemcq",
                                         _n.replace("livemcq_", "")),
                        required=False)
        if _p:
            _lm_paths.append(_p)
            break
    if _lm_paths:
        livemcq_src, _np, _dp = build_bank(_lm_paths, drop_open=False)
        livemcq_keys = list(livemcq_src)
        print(f"livemcq bank: {_np} pairs -> {len(livemcq_src)} keys "
              f"({_lm_paths[0]})")
    else:
        print(f"!!! livemcq bank NOT attached (looked for {LIVEMCQ_BANK_NAMES}) -> "
              f"the cb_livemcq2 tier is DISABLED (711/2516 rows on the public set)"
              f"\n    attach mdmeheduzzaman/bengali-halluc-source-banks")

    # sibling sites: flat `sites_<domain>_qa.json` x12 (Kaggle) or nested
    # `sites/<domain>/qa.json` (local dev). Both spellings are globbed.
    _site_paths = find_inputs(SITE_BANK_PATTERNS,
                              (os.path.join(_WORK, "source_hunt"), _WORK))
    if _site_paths:
        _doms = sorted({(SITE_FLAT_RE.match(os.path.basename(p)).group(1)
                         if SITE_FLAT_RE.match(os.path.basename(p))
                         else os.path.basename(os.path.dirname(p)))
                        for p in _site_paths})
        sites_src, _np, _dp = build_bank(_site_paths, drop_open=True)
        sites_keys = list(sites_src)
        print(f"sibling-site banks: {len(_site_paths)} files / {len(_doms)} domains, "
              f"{_np} pairs (dropped {_dp} open-ended) -> {len(sites_src)} keys")
        print(f"  domains: {_doms}")
    else:
        print(f"sibling-site banks not attached (looked for {SITE_BANK_PATTERNS}) "
              f"-> the cb_sites tier is disabled")

# ---- (c) bn_grammar_kb canon tier (work/gram_match.py) ----------------------
# Loaded through the module bridge so the shipped canon file is the one used.

# ---- run the closed-book source layers -------------------------------------
cb_gold, cb_livemcq, cb_sites, cb_gram = {}, {}, {}, {}
if LIVE:
    _t0 = time.time()
    _val = []
    for i in CB_S:
        p, _ = cb_gold_pred(S[i]["prompt_bn"], S[i]["response_bn"])
        if p is not None:
            _val.append((p, S[i]["label"]))
    if _val:
        print(f"cb_gold sample validation: predicted {len(_val)}/{len(CB_S)} labeled "
              f"cb rows, accuracy = {sum(p == y for p, y in _val)/len(_val):.3f}")

    for i in CB_T:
        if i in leak_pred:
            continue
        pr, rs = T[i]["prompt_bn"], T[i]["response_bn"]
        p, _ = cb_gold_pred(pr, rs)
        if p is not None:
            cb_gold[i] = p
        if livemcq_src:
            p2, _ = bank_predict(ocr_fix(pr), ocr_fix(rs), livemcq_src, livemcq_keys)
            if p2 is not None:
                cb_livemcq[i] = p2
        if sites_src:
            p3, _ = bank_predict(ocr_fix(pr), ocr_fix(rs), sites_src, sites_keys)
            if p3 is not None:
                cb_sites[i] = p3
        g = tier_call("gram_match", "match", pr, rs)
        if isinstance(g, (list, tuple)):
            g = g[0] if g else None
        if g in (0, 1):
            cb_gram[i] = int(g)
    print(f"cb_gold(mmlu)={len(cb_gold)}  cb_livemcq={len(cb_livemcq)}  "
          f"cb_sites={len(cb_sites)}  cb_gram={len(cb_gram)}  ({time.time()-_t0:.0f}s)")
    emit_submission(tag="(after closed-book source matching)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 11: HELD-OUT layer — symbolic math solver + residual lookup tier.
# work/math_solve.py recognizes 26 arithmetic word-problem templates by explicit
# anchor regexes, solves each in exact rational arithmetic (fractions.Fraction),
# and compares to the number the response asserts. It ABSTAINS on anything
# ambiguous. work/cb_last_tier.py adds nCr / day-of-week / sqrt /
# inscribed-angle solvers plus public-source and closed-set canon lookups.
cb_mathsolve, cb_last = {}, {}
MATH_T = []
if LIVE:
    _t0 = time.time()
    for i in CB_T:
        if i in leak_pred:
            continue
        q, a = T[i]["prompt_bn"], T[i]["response_bn"]
        v = tier_call("math_solve", "solve", q, a)
        if isinstance(v, (list, tuple)):
            v = v[0] if v else None
        if v in (0, 1):
            cb_mathsolve[i] = int(v)
            continue
        v2 = tier_call("cb_last_tier", "solve_math", q, a)
        if isinstance(v2, (list, tuple)):
            v2 = v2[0] if v2 else None
        if v2 in (0, 1):
            cb_last[i] = int(v2)
    print(f"cb_mathsolve (symbolic solver): {len(cb_mathsolve)} rows | "
          f"cb_last (residual solvers): {len(cb_last)} rows ({time.time()-_t0:.0f}s)")

    # Arithmetic rows the symbolic solver ABSTAINED on go to the thinking-mode
    # math router (Qwen3-8B). On the public set this is 11 rows; on a held-out
    # fold it scales with the arithmetic share of the data.
    MATH_T = [i for i in CB_T
              if i not in leak_pred and i not in cb_mathsolve and i not in cb_last
              and i not in cb_gold and i not in cb_livemcq
              and is_math_prompt(T[i]["prompt_bn"])]
    print(f"math router pool (solver abstained): {len(MATH_T)} rows")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 12: HELD-OUT layer — idiom / dictionary gloss tiers and the wiki-article
# tier. All CPU-only, all run before any model load.
#   * wikt   — bn.wiktionary gloss overlap (work/wikt_idioms.py)
#   * tail   — idiom tail-4 gloss overlap (work/idiom_tail4.py + assets)
#   * wsrc   — Bangla Academy dictionary, cache-only (work/wikisource_tier.py)
#   * wiki   — bn.wikipedia article grounding (work/cb_wiki_tier.py)
# Each runs only where its data files are attached; a missing file disables one
# tier and its rows fall through.
GLOSS_OVERLAP_THR = 0.34          # the validated shipped constant
cb_wikt, cb_tail, cb_wsrc, cb_wiki = {}, {}, {}, {}

WIKT_IDIOM_Q = re.compile(r'^["“]?(.+?)["”]?\s*এর\s*(ভাবার্থ|শাব্দিক অর্থ)\s*কী\s*\??$')


def wikt_extract_idiom(prompt):
    m = WIKT_IDIOM_Q.match(str(prompt).strip())
    if m:
        return m.group(1).strip().strip('"“”'), m.group(2)
    return None, None


def gl_content_tokens(s):
    STOP = {"করা", "হওয়া", "যে", "বা", ";", ","}
    return {t for t in re.findall(r"[ঀ-৿]+", str(s)) if len(t) > 1 and t not in STOP}


wikt_lut, ws_cache = {}, {}
if LIVE:
    _wp = find_input("wikt_pages.json", os.path.join(_WORK, "wikt_pages.json"),
                     required=False)
    if _wp:
        _pages = json.load(open(_wp, encoding="utf-8"))
        _lut = tier_call("wikt_idioms", "build_lookup", _pages)
        wikt_lut = _lut or {}
        print(f"gloss tier (bn.wiktionary): {len(wikt_lut)} entries")
    else:
        print("gloss tier (bn.wiktionary): wikt_pages.json not attached -> disabled")

    _wsc = find_input("wikisource_tier_cache.json",
                      os.path.join(_WORK, "source_data", "wikisource_tier_cache.json"),
                      required=False)
    if _wsc:
        ws_cache = json.load(open(_wsc, encoding="utf-8"))
        print(f"gloss tier (BA dictionary cache): {len(ws_cache)} cached queries")
    else:
        print("gloss tier (BA dictionary): cache not attached -> disabled")

    _t0 = time.time()
    _covered = set(leak_pred) | set(cb_gold) | set(cb_livemcq) | set(cb_sites) \
        | set(cb_gram) | set(cb_mathsolve) | set(cb_last)
    # All four tiers below score the SAME residual pool independently; they do
    # not skip each other's hits. Conflicts are resolved at assembly time by the
    # final_entry precedence (LAYER_ORDER_CB: cb_wiki > cb_tail > cb_idiom > cb_wikisource),
    # so the compute order here cannot change any prediction.

    # ---- tier: bn.wiktionary gloss overlap ----------------------------------
    for i in CB_T:
        if i in _covered:
            continue
        idiom, qtype = wikt_extract_idiom(T[i]["prompt_bn"])
        if idiom and wikt_lut:
            v = tier_call("wikt_idioms", "predict", idiom, qtype,
                          T[i]["response_bn"], wikt_lut)
            # wikt_idioms.predict returns (pred, how) — unwrap like every other
            # tier call, otherwise `v in (0, 1)` is False for EVERY row and the
            # tier silently never fires.
            if isinstance(v, (list, tuple)):
                v = v[0] if v else None
            if v in (0, 1):
                cb_wikt[i] = int(v)

    # ---- tier: idiom tail-4 (merged public gloss dictionary) ----------------
    # Entry points: idiom_tail2.merged_glosses() builds the phrase -> [gloss]
    # dictionary from the curated canon + the harvested bn.wiktionary dump;
    # idiom_tail2.build_rows() selects the idiom-family rows; idiom_tail4
    # annotate/decide apply the stem-overlap rule. TAIL4_THR / TAIL4_REUSE are
    # the shipped defaults (work/idiom_tail4.py main()).
    TAIL4_THR, TAIL4_REUSE = 0.50, 0.50
    _GL = tier_call("idiom_tail2", "merged_glosses")
    if _GL:
        _trows = tier_call("idiom_tail2", "build_rows", T) or []
        tier_call("idiom_tail4", "annotate", _trows, _GL, TAIL4_REUSE)
        for _row in _trows:
            _i = _row.get("i")
            if _i is None or _i in _covered:
                continue
            _d = tier_call("idiom_tail4", "decide", _row, TAIL4_THR)
            _v = _d[0] if isinstance(_d, (list, tuple)) and _d else None
            if _v in (0, 1):
                cb_tail[_i] = int(_v)
        print(f"  idiom tail-4: {len(_GL)} gloss phrases, {len(_trows)} family rows")

    # ---- tier: Bangla Academy dictionary, CACHE-ONLY (network disabled) ------
    if TIER_MODULES.get("wikisource_tier") is not None:
        for i in CB_T:
            if i in _covered:
                continue
            idiom, qtype = wikt_extract_idiom(T[i]["prompt_bn"])
            if not idiom:
                continue
            _g = tier_call("wikisource_tier", "lookup", idiom, qtype or "শাব্দিক অর্থ")
            _gloss = _g[0] if isinstance(_g, (list, tuple)) and _g else None
            if not _gloss:
                continue
            _v = tier_call("wikisource_tier", "predict", _gloss,
                           T[i]["response_bn"], GLOSS_OVERLAP_THR)
            # wikisource_tier.predict returns (pred, how) — same unwrap as the
            # lookup() two lines above; without it the tier never fires.
            if isinstance(_v, (list, tuple)):
                _v = _v[0] if _v else None
            if _v in (0, 1):
                cb_wsrc[i] = int(_v)

    # ---- tier: bn.wikipedia article grounding -------------------------------
    # GENERALIZATION FIX (Phase-2 risk 3). work/cb_wiki_tier.py maps rows to
    # articles through ROW_ARTICLE, a hardcoded span table keyed by PUBLIC-SET
    # row index (318-380). Those indices are meaningless on a held-out fold: row
    # 318 there is some unrelated question, and grounding it against
    # "কাজী নজরুল ইসলাম" would produce a confident wrong label. So there are
    # exactly two mutually exclusive paths, and which one ran is asserted and
    # printed:
    #
    #   PATH=ROW_ARTICLE_INDEX   only when the mounted ids are the reference
    #                            CSV's ids IN THE REFERENCE ORDER (and this is
    #                            not a duplicated scale test). This is the
    #                            public set, where the table is exactly what
    #                            shipped and reproduces the 62 final_entry rows.
    #   PATH=QUESTION_TEXT       every other case. The table is not merely
    #                            unused — `_rowmap` is empty, and the assertion
    #                            below fails the cell rather than let a single
    #                            index lookup through.
    #
    # QUESTION_TEXT resolution, and why the tier is KEPT rather than disabled on
    # held-out folds (measured on the 2,516 public rows against the final_entry
    # reference, treating the whole closed-book segment as if it were unseen):
    #
    #   resolver                              claimed  accuracy
    #   longest-title-wins (first draft)          72     0.917
    #   UNIQUE-title-required (shipped here)      66     0.939
    #   tier disabled, rows fall to cb_default    66     0.515
    #
    # The first draft's failures were systematic, not noise: for
    # a prompt naming both a 19th-c. Bengali novelist AND one of his novels, asking for the novel's publication year: both the AUTHOR
    # and the NOVEL are in the corpus, longest-title-wins picks the author, and
    # the publication year is then read off the wrong article. Requiring a
    # UNIQUE title hit turns every such ambiguous prompt into an abstention and
    # removes that failure mode entirely (5 mis-groundings -> 0).
    #
    # Disabling the tier is strictly worse. Its residual errors are all
    # false-hallucination calls (predicts 0 where the truth is 1) — the SAME
    # direction as the final_entry closed-book default that would catch those rows if
    # the tier abstained. So switching it off does not fix them; it only gives
    # up the 62 rows it gets right, dropping 0.939 to 0.515 on the rows in
    # question. KEEP, with the unique-title guard.
    _wiki_dir = (find_dir("wiki_articles",
                          os.path.join(TIER_CODE_DIR or _WORK, "wiki_articles"))
                 or find_dir("wiki_articles", os.path.join(_WORK, "wiki_articles")))
    if _wiki_dir:
        _arts = {}
        for _p in sorted(glob.glob(os.path.join(_wiki_dir, "*.txt"))):
            _title = os.path.splitext(os.path.basename(_p))[0]
            try:
                _arts[_title] = open(_p, encoding="utf-8").read()
            except Exception:
                pass
        # core name for matching: drop the disambiguating parenthetical and the
        # '_' separator this corpus uses for slashes in titles
        _cores = [(re.sub(r"\s*\([^)]*\)", "", t).replace("_", " ").strip(), t)
                  for t in _arts]
        _cores = [(hunt_norm(c), t) for c, t in _cores if len(c) >= 4]

        def _resolve_article(prompt):
            """The article named by the prompt, or None. A prompt naming TWO or
            more corpus articles is ambiguous (author + work) and abstains."""
            _p = hunt_norm(prompt)
            hits = {t for c, t in _cores if c and c in _p}
            return next(iter(hits)) if len(hits) == 1 else None

        _public_order = (REF_ORDER is not None and not MAKE_SCALE_TEST
                         and REF_ORDER == [str(r["id"]).strip() for r in T])
        _rowmap = (getattr(TIER_MODULES.get("cb_wiki_tier"), "ROW_ARTICLE", {})
                   if _public_order else {})
        # HARD GUARD: the index table may exist in memory ONLY on the confirmed
        # public row order. Anything else and it must be empty here.
        assert _public_order or not _rowmap, (
            "cb_wiki ROW_ARTICLE index table is populated on a non-public row "
            "order — this would ground held-out rows against unrelated articles")
        if _rowmap:
            print(f"  cb_wiki: PATH=ROW_ARTICLE_INDEX — public row order confirmed "
                  f"against the reference CSV; using the shipped index table "
                  f"({len(_rowmap)} rows, reproduces the 62 final_entry cb_wiki rows)")
        else:
            print(f"  cb_wiki: PATH=QUESTION_TEXT — held-out row order; the "
                  f"ROW_ARTICLE index table is NOT consulted (len={len(_rowmap)}). "
                  f"Articles resolved from the prompt over {len(_arts)} titles, "
                  f"unique-title-hit required (ambiguous prompts abstain).")
        _n_index, _n_text = 0, 0
        for i in CB_T:
            if i in _covered:
                continue
            _text = None
            if i in _rowmap:
                _text = tier_call("cb_wiki_tier", "article_text", _rowmap[i])
                _n_index += 1
            else:
                _t = _resolve_article(T[i]["prompt_bn"])
                if _t:
                    _text = _arts[_t]
                    _n_text += 1
            if not _text:
                continue
            _d = tier_call("cb_wiki_tier", "classify", T[i]["prompt_bn"],
                           T[i]["response_bn"], _text)
            _v = _d[0] if isinstance(_d, (list, tuple)) and _d else None
            if _v in (0, 1):
                cb_wiki[i] = int(_v)
        # SAFETY INVARIANT. The only thing that must hold is that the index
        # table is never consulted on a non-public row order — asserted below.
        # The two counters are NOT mutually exclusive: on the public order the
        # index table covers 63 of the ~1,155 closed-book rows and every other
        # row still resolves through the question-text path, so both are nonzero
        # by design (measured: 63 index + 6 text). A previous
        # `assert not (_n_index and _n_text)` here raised AssertionError and
        # aborted the whole live pipeline on the public set — i.e. exactly the
        # fallback that runs if the artifacts dataset fails to attach — leaving
        # the run stranded on the crash-safety CSV with every closed-book
        # residual defaulted and no LLM judge ever reached.
        assert _public_order or _n_index == 0, (
            f"cb_wiki used the ROW_ARTICLE index on {_n_index} held-out rows")
        print(f"  cb_wiki: grounded {_n_index} rows via the index table and "
              f"{_n_text} via question text -> {len(cb_wiki)} predictions "
              f"(path taken: "
              f"{'ROW_ARTICLE_INDEX' if _n_index else 'QUESTION_TEXT'})")
    else:
        print("  cb_wiki: wiki_articles/ not attached -> tier disabled")

    print(f"gloss/wiki tiers: wikt={len(cb_wikt)} tail={len(cb_tail)} "
          f"wsrc={len(cb_wsrc)} wiki={len(cb_wiki)} ({time.time()-_t0:.0f}s)")
    emit_submission(tag="(after gloss + wiki tiers)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 13: HELD-OUT judge pools + governor plan.
# Whatever survives every deterministic layer goes to the LLM judges. This is
# the pool whose size the governor must control: on the public set it is 234
# rows; on a held-out fold where few rows match a source it can be thousands.
CTX_JUDGE_T, CB_JUDGE_T = [], []
if LIVE:
    CTX_JUDGE_T = [i for i in CTX_T
                   if i not in leak_pred and i not in ctx_gold and i not in ctx_det]
    _cb_covered = (set(leak_pred) | set(cb_gold) | set(cb_livemcq) | set(cb_sites)
                   | set(cb_gram) | set(cb_mathsolve) | set(cb_last) | set(cb_wikt)
                   | set(cb_tail) | set(cb_wsrc) | set(cb_wiki) | set(MATH_T))
    CB_JUDGE_T = [i for i in CB_T if i not in _cb_covered]
    print(f"judge pools: ctx_think={len(CTX_JUDGE_T)}  math_router={len(MATH_T)}  "
          f"cb_residual(32B)={len(CB_JUDGE_T)}")
    print(f"deterministic coverage: "
          f"{len(T) - len(CTX_JUDGE_T) - len(MATH_T) - len(CB_JUDGE_T)}/{len(T)} rows "
          f"({100*(len(T)-len(CTX_JUDGE_T)-len(MATH_T)-len(CB_JUDGE_T))/max(len(T),1):.1f}%)")

    governor.set_plan(math=len(MATH_T), ctx_think=len(CTX_JUDGE_T))
    # Reserve the projected 32B stage so the ctx ladder degrades early enough to
    # leave room for it. Zeroed when the 32B stage starts or is skipped.
    if J32_ENABLED and CB_JUDGE_T:
        governor.extra_pending_s = (J32_LOAD_EST_S + len(CB_JUDGE_T) * J32_SROW_EST)
        print(f"32B residual judge reserved: {len(CB_JUDGE_T)} rows, "
              f"{governor.extra_pending_s/60:.0f} min held back in the budget model")

    _proj0 = governor.project()
    print(f"pre-flight projection at {governor.level_key()}: "
          f"{_proj0/3600:.2f}h (soft {SOFT_BUDGET_H}h, hard {HARD_CAP_H}h)")
    while _proj0 > SOFT_BUDGET_S and governor.level < len(governor.LEVELS) - 1:
        governor._degrade_once(f"PRE-FLIGHT projection {_proj0/3600:.2f}h > soft "
                               f"budget {SOFT_BUDGET_H}h")
        _proj0 = governor.project()



In [ ]:
# %% ------------------------------------------------------------------
# Cell 14: Judge model load — Qwen3-8B, 4-bit NF4.
# WEIGHTS BUDGET (Phase-2 cap 50 GB): Qwen3-8B ~5 GB as a 4-bit mirror, or
# 16.4 GB of fp16 safetensors quantized at load. Optional Qwen3-32B 4-bit
# ~18 GB. Worst case ~34.4 GB < 50 GB. PASS.
# The two models are NEVER resident at the same time — see free_model (Cell 2).
tokenizer = model = None
YES_ID = NO_ID = -1
W8_GB = 0.0                    # measured on-disk size of the attached 8B weights


def _variant_of(cfg_path, cfg):
    """'8b' | '32b' | None for a Qwen3 config — by ARCHITECTURE, not by name.
    Qwen3-8B is 36 layers / hidden 4096; Qwen3-32B is 64 layers / hidden 5120.
    Falls back to a path token when the config omits the shape."""
    n = cfg.get("num_hidden_layers")
    h = cfg.get("hidden_size")
    if isinstance(n, int) and n > 0:
        return "32b" if n > 48 else "8b"
    if isinstance(h, int) and h > 0:
        return "32b" if h >= 5120 else "8b"
    p = cfg_path.lower()
    if "32b" in p:
        return "32b"
    if "8b" in p:
        return "8b"
    return None


def resolve_model_dir(candidates, want="qwen3", variant=None):
    """Locate a model directory. `variant` ('8b'/'32b') constrains the SEARCH
    FALLBACK.

    WHY THE CONSTRAINT EXISTS. The fallback scans every config.json under
    /kaggle/input in SORTED order and returns the first Qwen3 it finds. Kaggle's
    official `qwen-3` model dataset ships both variants side by side, and
    '.../transformers/32b/1' sorts BEFORE '.../transformers/8b/<v>' because
    '3' < '8'. So if the attached 8B sits at a version directory not listed in
    MODEL_CANDIDATES (Kaggle bumps those: /1 -> /2 -> /3 ...), load_8b() would
    silently resolve the THIRTY-TWO BILLION parameter model — a wrong, far
    slower, possibly non-fitting load, with the 8B judge never running. The
    reverse swap is equally possible for the 32B stage. Explicit candidate paths
    are still honoured first and unchanged.
    """
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    for cfg_path in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            with open(cfg_path) as f:
                cfg = json.load(f)
            if want not in str(cfg.get("model_type", "")).lower():
                continue
            if variant is not None:
                v = _variant_of(cfg_path, cfg)
                if v is not None and v != variant:
                    print(f"  (skipping {os.path.dirname(cfg_path)}: looks like "
                          f"Qwen3-{v.upper()}, wanted {variant.upper()})")
                    continue
            return os.path.dirname(cfg_path)
        except Exception:
            pass
    return None


# --- WEIGHTS BUDGET AUDIT (Phase-2 cap: 50 GB of model weights) --------------
# Measured, not asserted in a comment. The cap is about what is ATTACHED, so it
# is computed from the on-disk shards of whatever resolve_model_dir found.
# This matters: MODEL32_CANDIDATES lists Kaggle's official Qwen-3 model dataset,
# whose 32B variant ships as ~65 GB of fp16 safetensors. Attaching THAT instead
# of a pre-quantized 4-bit mirror would blow the 50 GB cap on its own even
# though it quantizes down to ~18 GB in VRAM at load. So the 32B stage is
# skipped, loudly, when the attached weights do not fit the budget.
WEIGHTS_CAP_GB = 50.0
_WEIGHT_EXT = (".safetensors", ".bin", ".pt", ".pth", ".gguf")


def weights_gb(d):
    """Total on-disk size (GB) of the model shards in directory `d`."""
    if not d or not os.path.isdir(d):
        return 0.0
    n = 0
    for root, _, files in os.walk(d):
        for f in files:
            if f.endswith(_WEIGHT_EXT):
                try:
                    n += os.path.getsize(os.path.join(root, f))
                except OSError:
                    pass
    return n / 1e9


def load_8b():
    """Load Qwen3-8B 4-bit. Returns True on success."""
    global tokenizer, model, YES_ID, NO_ID
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    d = resolve_model_dir(MODEL_CANDIDATES, variant="8b") or LOCAL_MODEL_FALLBACK
    global W8_GB
    W8_GB = weights_gb(d)
    print(f"8B model: {d}  ({W8_GB:.1f} GB of weights on disk, "
          f"cap {WEIGHTS_CAP_GB:.0f} GB)")
    t = time.time()
    tokenizer = AutoTokenizer.from_pretrained(d)
    tokenizer.padding_side = "left"        # last position == generation point
    tokenizer.truncation_side = "left"     # keep the QA + final instruction
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,  # T4/P100: no bf16
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        d, quantization_config=bnb, torch_dtype=torch.float16, device_map="auto")
    model.eval()
    YES_ID = tokenizer.encode("Yes", add_special_tokens=False)[0]
    NO_ID = tokenizer.encode("No", add_special_tokens=False)[0]
    assert YES_ID != NO_ID
    print(f"8B loaded in {time.time()-t:.0f}s (elapsed {governor.elapsed()/60:.1f}m)  "
          f"YES_ID={YES_ID} NO_ID={NO_ID}")
    return True


NEED_8B = bool(LIVE and THINKING_MODE_ENABLED and governor.mode() != "rule"
               and (CTX_JUDGE_T or MATH_T))
if NEED_8B:
    try:
        load_8b()
    except Exception as e:
        print(f"!!! 8B load FAILED ({type(e).__name__}: {e}) -> the judge ladder "
              f"drops to the substring rule; the run still produces a valid CSV")
        governor.level = len(governor.LEVELS) - 1
        governor.events.append("8B load failed -> forced T2_substring")
        tokenizer = model = None
elif LIVE:
    print("8B judge load SKIPPED "
          + ("(governor pre-flight already at T2_substring)"
             if governor.mode() == "rule" else "(no rows to judge)"))
else:
    print("8B judge load SKIPPED — public-rerun fast path")

EMPTY_THINK = "<think>\n\n</think>\n\n"


def chat_text(user_content):
    """Chat-templated prompt whose NEXT token should be Yes/No (thinking off)."""
    msgs = [{"role": "user", "content": user_content}]
    try:
        s = tokenizer.apply_chat_template(msgs, tokenize=False,
                                          add_generation_prompt=True,
                                          enable_thinking=False)
    except TypeError:                       # older transformers
        s = tokenizer.apply_chat_template(msgs, tokenize=False,
                                          add_generation_prompt=True)
        if not s.rstrip().endswith("</think>"):
            s += EMPTY_THINK
    return s


@torch.inference_mode()
def logprob_batch(texts, max_len=None):
    """P(Yes) per text — one left-padded prefill forward pass, no generation."""
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                    max_length=max_len or MAX_TOKENS, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    logits = model(**enc).logits[:, -1, :].float()
    p = torch.softmax(torch.stack([logits[:, YES_ID], logits[:, NO_ID]], -1), -1)[:, 0]
    return p.cpu().tolist()


@torch.inference_mode()
def generate_verdict_text(user_content, max_new_tokens):
    """Thinking-mode greedy generation; returns the decoded assistant text."""
    msgs = [{"role": "user", "content": user_content}]
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True,
                                             enable_thinking=True)
    except TypeError:                       # older transformers: thinking default-on
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=MAX_TOKENS, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


# ---- prompts, ported verbatim from work/math_router.py and work/ctx_think.py --
MATH_THINK_PROMPT = """নিচের গণিত প্রশ্নটি নিজে সমাধান করো, তারপর প্রদত্ত উত্তরটি যাচাই করো।

প্রশ্ন: {prompt}
প্রদত্ত উত্তর: {response}

Solve it yourself step by step, then compare. End your reply with exactly one line: "VERDICT: Yes" if the given answer is correct, or "VERDICT: No" if it is wrong."""

CTX_THINK_PROMPT = """অনুচ্ছেদ:
{context}

প্রশ্ন: {prompt}
প্রদত্ত উত্তর: {response}

Task: decide whether the given answer is FAITHFUL to the passage.

Rules:
- Judge ONLY against the passage. Ignore whether it is true in the real world.
- The answer may paraphrase, abbreviate (বিএনপি = বাংলাদেশ জাতীয়তাবাদী দল), or inflect words — that is still faithful if the meaning is supported.
- An answer that is incomplete but correct as far as it goes is FAITHFUL.
- It is HALLUCINATED if any name, date, number, or place contradicts the passage, or if it states a fact the passage does not support.
- Watch for numbers that look similar (৭৭৬ vs ৭৭৪) and for entities that appear in the passage but in a different role.

Be brief. Then end with exactly one line:
VERDICT: Faithful
or
VERDICT: Hallucinated"""

# One-token logprob prompt — the T1 rung of the ladder.
CTX_LOGPROB_PROMPT = """অনুচ্ছেদ:
{context}

প্রশ্ন: {prompt}
প্রদত্ত উত্তর: {response}

Judge ONLY against the passage above; ignore whether it is true in the real world. Is the given answer faithful to the passage? A name, date, number, or place that contradicts the passage — or a claim the passage does not support — makes it unfaithful. Reply with exactly one word, "Yes" (faithful) or "No" (hallucinated)."""

CB_LOGPROB_PROMPT = """তুমি একজন বাংলা ভাষার তথ্য যাচাইকারী। নিচের প্রশ্ন এবং উত্তর দেখো।

প্রশ্ন: {prompt}
উত্তর: {response}

Based on your knowledge, is the answer factually correct? A wrong fact, wrong person, wrong date, or fabricated detail means it is not correct. Reply with exactly one word, "Yes" or "No"."""


def math_judge(r):
    out = generate_verdict_text(
        MATH_THINK_PROMPT.format(prompt=r["prompt_bn"], response=r["response_bn"]),
        MATH_MAX_NEW_TOKENS)
    m = re.findall(r"VERDICT:\s*(Yes|No)", out)
    if m:
        return 1 if m[-1] == "Yes" else 0
    return 0 if re.search(r"\bNo\b|ভুল|সঠিক নয়", out[-200:]) else 1


def ctx_think_judge(r):
    out = generate_verdict_text(
        CTX_THINK_PROMPT.format(context=str(r["context"])[:3000],
                                prompt=r["prompt_bn"], response=r["response_bn"]),
        CTX_MAX_NEW_TOKENS)
    m = re.findall(r"VERDICT:\s*(Faithful|Hallucinated)", out)
    if m:
        return 1 if m[-1] == "Faithful" else 0
    return 0 if re.search(r"Hallucinat|ভুল|সমর্থিত নয়", out[-300:]) else 1




In [ ]:
# %% ------------------------------------------------------------------
# Cell 15: HELD-OUT — the governed judge stages.
#   (a) math router: few rows, always the most capable available mode.
#   (b) ctx judge: warmup -> project -> thinking, or degrade to logprob, or
#       (worst case) leave the rest to the substring rule.
math_pred, ctx_think_pred = {}, {}


def ctx_logprob_pass(idxs):
    """T1 rung: one prefill Yes/No pass per row. Hard-cap guarded. Any row left
    unscored falls through to the substring rule at assembly."""
    if not idxs or model is None:
        return
    texts = [chat_text(CTX_LOGPROB_PROMPT.format(
        context=str(T[i]["context"])[:3000], prompt=T[i]["prompt_bn"],
        response=T[i]["response_bn"])) for i in idxs]
    order = sorted(range(len(idxs)), key=lambda j: len(texts[j]))  # min padding waste
    bs, j, t0 = BATCH_SIZE, 0, time.time()
    while j < len(order):
        if governor.elapsed() > HARD_CAP_S:
            governor.tripped = True
            print("  [ctx-logprob] HARD CAP -> remaining rows use the substring rule",
                  flush=True)
            break
        chunk = order[j:j + bs]
        try:
            pys = logprob_batch([texts[t] for t in chunk])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            print(f"  [ctx-logprob] CUDA OOM -> batch {bs}", flush=True)
            continue
        for t, py in zip(chunk, pys):
            ctx_think_pred[idxs[t]] = 1 if py > 0.5 else 0
        j += len(chunk)
        governor.consume("ctx_think", len(chunk))
        # T1 -> T2 RUNG. If the governor degrades HERE, the logprob rung itself
        # is too slow for the remaining pool and this pass must STOP, handing
        # the rest to the parameter-free substring rule. Without the break the
        # pass ran on to the 8.25 h hard fuse: simulated at 12 s/row over 4,000
        # rows it burned the entire budget, scored only 2,176/4,000 and starved
        # the 32B stage; breaking here finishes the same case in 1.05 h.
        if governor.checkpoint("ctx-logprob", j, len(order), t0) == "degrade":
            print(f"  [ctx-logprob] governor degraded to {governor.level_key()} "
                  f"-> stopping the logprob pass at {j}/{len(order)}; the "
                  f"remaining {len(order)-j} rows use the substring rule",
                  flush=True)
            break


if LIVE and model is not None:
    # ---- (a) math router --------------------------------------------------
    if MATH_T:
        print(f"=== math router: {len(MATH_T)} rows (mode={governor.mode()}) ===",
              flush=True)
        t_m = time.time()
        for k, i in enumerate(MATH_T):
            if governor.tripped or governor.elapsed() > HARD_CAP_S:
                governor.tripped = True
                print("!!! HARD CAP during the math router -> remaining math rows "
                      "fall to the closed-book default", flush=True)
                break
            if governor.mode() == "thinking":
                math_pred[i] = math_judge(T[i])
            elif governor.mode() == "logprob":
                py = logprob_batch([chat_text(CB_LOGPROB_PROMPT.format(
                    prompt=T[i]["prompt_bn"], response=T[i]["response_bn"]))])[0]
                math_pred[i] = 1 if py > 0.5 else 0
            else:
                break
            governor.consume("math", 1)
            governor.checkpoint("math", k + 1, len(MATH_T), t_m,
                                force=((k + 1) % 10 == 0 or k + 1 == len(MATH_T)))
        emit_submission(tag="(after the math router)")

    # ---- (b) ctx judge: warmup -> project -> thinking or degrade ----------
    if CTX_JUDGE_T:
        print(f"=== ctx judge: {len(CTX_JUDGE_T)} rows (start mode="
              f"{governor.mode()}) ===", flush=True)
        if governor.mode() == "thinking":
            warm = CTX_JUDGE_T[:THINK_WARMUP_ROWS]
            t_w = time.time()
            for i in warm:
                if governor.elapsed() > HARD_CAP_S:
                    governor.tripped = True
                    break
                ctx_think_pred[i] = ctx_think_judge(T[i])
                governor.consume("ctx_think", 1)
            n_warm = sum(1 for i in warm if i in ctx_think_pred)
            if n_warm:
                governor.rate["thinking"] = (time.time() - t_w) / n_warm
            proj = governor.project()
            print(f"[ctx-think] warmup {n_warm} rows @ "
                  f"{governor.srow('thinking'):.1f}s/row -> projected_total="
                  f"{proj/3600:.2f}h  (soft {SOFT_BUDGET_H}h, "
                  f"{len(CTX_JUDGE_T)-n_warm} rows remaining)", flush=True)
            while proj > SOFT_BUDGET_S and governor.level < len(governor.LEVELS) - 1:
                governor._degrade_once(f"post-warmup projection {proj/3600:.2f}h > "
                                       f"soft budget {SOFT_BUDGET_H}h")
                proj = governor.project()
            rest = [i for i in CTX_JUDGE_T if i not in ctx_think_pred]
        else:
            rest = list(CTX_JUDGE_T)

        if governor.mode() == "thinking" and not governor.tripped:
            t_c = time.time()
            for k, i in enumerate(rest):
                if governor.elapsed() > HARD_CAP_S:
                    governor.tripped = True
                    print("!!! HARD CAP during the ctx judge -> remaining rows use "
                          "the substring rule", flush=True)
                    break
                ctx_think_pred[i] = ctx_think_judge(T[i])
                governor.consume("ctx_think", 1)
                if governor.checkpoint("ctx-think", k + 1, len(rest), t_c) == "degrade":
                    # mid-pass degradation: hand the remainder to the cheap rung
                    if governor.mode() == "logprob":
                        ctx_logprob_pass(rest[k + 1:])
                    break
        elif governor.mode() == "logprob" and not governor.tripped:
            ctx_logprob_pass(rest)

        _scored = sum(1 for i in CTX_JUDGE_T if i in ctx_think_pred)
        print(f"[ctx-judge] scored {_scored}/{len(CTX_JUDGE_T)} "
              f"(final mode={governor.mode()}; unscored -> substring rule)", flush=True)
        emit_submission(tag="(after the ctx judge)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 16: HELD-OUT — optional Qwen3-32B closed-book residual judge (cb_32B).
# Requirement 4: the 8B is FREED before the 32B is loaded, so the two never
# occupy VRAM together (2xT4 = 2x15 GB cannot hold both).
# Fully degradable: no weights attached, a load failure, a governor projection
# breach, or the hard fuse all skip this stage and leave the residual rows on
# the final_entry closed-book default (0 = hallucinated).
J32_MIN_VRAM_GIB = 20.0   # NF4 Qwen3-32B is ~18 GB of weights + activations/KV


def total_vram_gib():
    """Total VRAM across every visible GPU; 0.0 on CPU."""
    if not torch.cuda.is_available():
        return 0.0
    return sum(torch.cuda.get_device_properties(i).total_memory
               for i in range(torch.cuda.device_count())) / 2 ** 30


j32_pred = {}
if LIVE and J32_ENABLED and CB_JUDGE_T and not governor.tripped:
    _m32 = resolve_model_dir(MODEL32_CANDIDATES, variant="32b")
    _proj32 = (governor.elapsed() + J32_LOAD_EST_S
               + len(CB_JUDGE_T) * J32_SROW_EST + THINK_PROJ_MARGIN_S)
    _vram, _w32 = total_vram_gib(), weights_gb(_m32)
    governor.extra_pending_s = 0.0          # the reservation is now the real cost
    if _m32 is None:
        print("32B residual judge SKIPPED: no Qwen3-32B weights attached -> the "
              f"{len(CB_JUDGE_T)} residual rows take the final_entry default "
              f"({V23_CB_DEFAULT})")
    elif W8_GB + _w32 > WEIGHTS_CAP_GB:
        # Phase-2 attaches at most 50 GB of weights. A pre-quantized 4-bit 32B
        # mirror is ~18 GB and clears the cap alongside the 8B; the official
        # fp16 32B shards (~65 GB, and MODEL32_CANDIDATES does list Kaggle's
        # qwen-3 model dataset) do not, even though they quantize down to ~18 GB
        # in VRAM at load. Measure what is attached rather than assuming.
        print(f"32B residual judge SKIPPED: WEIGHTS BUDGET — 8B {W8_GB:.1f} GB + "
              f"32B {_w32:.1f} GB = {W8_GB + _w32:.1f} GB exceeds the "
              f"{WEIGHTS_CAP_GB:.0f} GB Phase-2 cap. Attach a 4-bit Qwen3-32B "
              f"mirror (~18 GB), not the fp16 shards. The {len(CB_JUDGE_T)} "
              f"residual rows take the final_entry default ({V23_CB_DEFAULT}).")
        governor.events.append(f"32B stage skipped (weights {W8_GB + _w32:.1f} GB "
                               f"> {WEIGHTS_CAP_GB:.0f} GB cap)")
    elif _vram < J32_MIN_VRAM_GIB:
        # A SINGLE P100 (16 GiB) CANNOT HOLD NF4 Qwen3-32B (~18 GB of weights).
        # device_map="auto" would try to spill onto CPU/disk, which bitsandbytes
        # 4-bit refuses — several minutes of shard reading before it raises, or
        # (worse, if accelerate does dispatch it) an unusably slow offloaded
        # run that the governor cannot see because this stage is off the ladder.
        # Decide it up front from the reported VRAM. The 2xT4 option
        # (2 x 15 = 30 GiB) clears this check and runs the stage normally.
        print(f"32B residual judge SKIPPED: only {_vram:.1f} GiB VRAM across "
              f"{torch.cuda.device_count()} gpu(s), need >= {J32_MIN_VRAM_GIB:.0f} "
              f"GiB for NF4 Qwen3-32B -> the {len(CB_JUDGE_T)} residual rows take "
              f"the final_entry default ({V23_CB_DEFAULT}). Select 2xT4 to enable it.")
        governor.events.append(f"32B stage skipped ({_vram:.1f} GiB VRAM available)")
    elif _proj32 > SOFT_BUDGET_S:
        print(f"32B residual judge SKIPPED: projected {_proj32/3600:.2f}h > soft "
              f"budget {SOFT_BUDGET_H}h")
        governor.events.append(f"32B stage skipped (projected {_proj32/3600:.2f}h)")
    else:
        print(f"=== 32B residual judge: {len(CB_JUDGE_T)} rows, model {_m32} ===",
              flush=True)
        free_model("model", "tokenizer")     # requirement 4: sequential loading
        try:
            from transformers import (AutoTokenizer, AutoModelForCausalLM,
                                      BitsAndBytesConfig)
            _t = time.time()
            tokenizer = AutoTokenizer.from_pretrained(_m32)
            tokenizer.padding_side = "left"
            tokenizer.truncation_side = "left"
            if tokenizer.pad_token_id is None:
                tokenizer.pad_token = tokenizer.eos_token
            _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                      bnb_4bit_compute_dtype=torch.float16,
                                      bnb_4bit_use_double_quant=True)
            model = AutoModelForCausalLM.from_pretrained(
                _m32, quantization_config=_bnb, torch_dtype=torch.float16,
                device_map="auto")
            model.eval()
            YES_ID = tokenizer.encode("Yes", add_special_tokens=False)[0]
            NO_ID = tokenizer.encode("No", add_special_tokens=False)[0]
            print(f"32B loaded in {time.time()-_t:.0f}s "
                  f"(elapsed {governor.elapsed()/60:.1f}m)", flush=True)

            _texts = [chat_text(CB_LOGPROB_PROMPT.format(
                prompt=T[i]["prompt_bn"], response=T[i]["response_bn"]))
                for i in CB_JUDGE_T]
            _order = sorted(range(len(CB_JUDGE_T)), key=lambda j: len(_texts[j]))
            _bs, _j, _t0, _since = J32_BATCH_CB, 0, time.time(), 0
            while _j < len(_order):
                if governor.elapsed() > HARD_CAP_S:
                    governor.tripped = True
                    print("  [32B] HARD CAP -> remaining residual rows take the "
                          "final_entry default", flush=True)
                    break
                _chunk = _order[_j:_j + _bs]
                try:
                    _pys = logprob_batch([_texts[t] for t in _chunk],
                                         max_len=J32_MAX_TOKENS)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    _bs = max(1, _bs // 2)
                    print(f"  [32B] CUDA OOM -> batch {_bs}", flush=True)
                    continue
                for t, py in zip(_chunk, _pys):
                    # P(halluc) = 1 - P(Yes); the shipped rule is P(halluc) > 0.50
                    j32_pred[CB_JUDGE_T[t]] = 0 if (1.0 - py) > J32_THR else 1
                _j += len(_chunk)
                _since += len(_chunk)
                if _since >= J32_EMPTY_CACHE_EVERY:
                    torch.cuda.empty_cache()
                    _since = 0
                # SOFT-BUDGET GUARD. This stage sits OFF the judge ladder, so
                # the governor cannot degrade it — without this its only fuse
                # was the 8.25 h hard cap, i.e. a measured s/row worse than the
                # J32_SROW_EST prior could eat the entire remaining budget.
                # Re-project from the MEASURED rate and stop cleanly instead;
                # the unscored residual rows take the final_entry default, which is
                # exactly what skipping the stage outright would have done.
                _r = (time.time() - _t0) / max(_j, 1)
                if (governor.elapsed() + (len(_order) - _j) * _r
                        + THINK_PROJ_MARGIN_S) > SOFT_BUDGET_S:
                    print(f"  [32B] measured {_r:.2f}s/row -> finishing the "
                          f"remaining {len(_order)-_j} rows would breach the "
                          f"{SOFT_BUDGET_H}h soft budget; stopping at "
                          f"{_j}/{len(_order)} (rest take the final_entry default)",
                          flush=True)
                    governor.events.append(
                        f"32B stage stopped early at {_j}/{len(_order)} "
                        f"({_r:.2f}s/row)")
                    break
                if _j % (J32_BATCH_CB * 10) < _bs:
                    print(f"  [32B] {_j}/{len(_order)}  {_r:.2f}s/row  "
                          f"elapsed={governor.elapsed()/3600:.2f}h", flush=True)
            print(f"32B residual judge: scored {len(j32_pred)}/{len(CB_JUDGE_T)} rows",
                  flush=True)
        except Exception as e:
            print(f"!!! 32B stage FAILED ({type(e).__name__}: {e}) -> residual rows "
                  f"take the final_entry default; the run continues")
            governor.events.append(f"32B stage failed: {type(e).__name__}")
        finally:
            free_model("model", "tokenizer")
    emit_submission(tag="(after the 32B residual judge)")
elif LIVE:
    print(f"32B residual judge not run (enabled={J32_ENABLED}, "
          f"rows={len(CB_JUDGE_T)}, fuse_tripped={governor.tripped})")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 17: Assemble predictions.
# HELD-OUT MODE walks the SAME precedence as the final_entry router (Cell 5), with the
# live-computed tiers substituting for the precomputed artifacts.


def live_lookup(i, r):
    """final_entry precedence over the live layers; None if nothing has decided row i."""
    G = globals()
    if i in G.get("leak_pred", {}):
        return G["leak_pred"][i]
    if r["context"]:
        for d in ("ctx_gold", "ctx_last_bip", "ctx_last_gloss", "ctx_det",
                  "ctx_think_pred"):
            m = G.get(d, {})
            if i in m:
                return m[i]
        return G.get("ctx_sub", {}).get(i)
    for d in ("cb_mathsolve", "cb_last", "cb_livemcq", "cb_gold", "math_pred",
              "cb_wiki", "cb_tail", "cb_wikt", "cb_wsrc", "cb_sites", "cb_gram",
              "j32_pred"):
        m = G.get(d, {})
        if i in m:
            return m[i]
    return None


LAYER_ORDER_CTX = [("leak_pred", "leak"), ("ctx_gold", "ctx_gold"),
                   ("ctx_last_bip", "ctx_LAST_BIPARIT"),
                   ("ctx_last_gloss", "ctx_LAST_GLOSS"),
                   ("ctx_det", "ctx_deterministic"), ("ctx_think_pred", "ctx_think")]
LAYER_ORDER_CB = [("leak_pred", "leak"), ("cb_mathsolve", "cb_mathsolve"),
                  ("cb_last", "cb_last"), ("cb_livemcq", "cb_livemcq2"),
                  ("cb_gold", "cb_gold"), ("math_pred", "cb_math"),
                  ("cb_wiki", "cb_wiki"), ("cb_tail", "cb_tail"),
                  ("cb_wikt", "cb_idiom"), ("cb_wsrc", "cb_wikisource"),
                  ("cb_sites", "cb_sites"), ("cb_gram", "cb_gram"),
                  ("j32_pred", "cb_32B")]

pred, layer_of, n = [], [], collections.Counter()
if FAST_PATH:
    for i, r in enumerate(T):
        pred.append(final_pred[i])
        layer_of.append(final_layer[i])
        n[final_layer[i]] += 1
else:
    G = globals()
    for i, r in enumerate(T):
        p, lay = None, None
        for name, label in (LAYER_ORDER_CTX if r["context"] else LAYER_ORDER_CB):
            m = G.get(name, {})
            if i in m:
                p, lay = m[i], label
                break
        if p is None:
            if r["context"]:
                p, lay = ctx_sub.get(i, 1), "ctx_substring"
            else:
                p, lay = V23_CB_DEFAULT, "cb_default"
        pred.append(int(p))
        layer_of.append(lay)
        n[lay] += 1

print("layers:", dict(sorted(n.items())))
print(f"pred distribution: halluc(0)={pred.count(0)}  faithful(1)={pred.count(1)}")
_write_submission(pred)
print("wrote", SUBMISSION_PATH)



In [ ]:
# %% ------------------------------------------------------------------
# Cell 18: Runtime report
print(f"\nTOTAL notebook time: {governor.elapsed()/3600:.2f}h "
      f"(platform limit {KAGGLE_LIMIT_H}h, soft {SOFT_BUDGET_H}h, "
      f"hard fuse {HARD_CAP_H}h, tripped={governor.tripped})")
print(f"final judge level: {governor.level_key()} (mode={governor.mode()})")
print(f"measured rates: "
      + ", ".join(f"{k}={v:.2f}s/row" for k, v in governor.rate.items()) or "none")
if governor.events:
    print(f"governor degradations ({len(governor.events)}):")
    for e in governor.events:
        print(f"  - {e}")
else:
    print("governor degradations: none (full-quality judges throughout)")



In [ ]:
# %% ------------------------------------------------------------------
# Cell 19: REPRODUCE_CHECK — row-by-row diff against the attached final_entry reference.
if not REPRODUCE_CHECK:
    print("REPRODUCE_CHECK disabled — diff skipped")
elif MAKE_SCALE_TEST:
    print("MAKE_SCALE_TEST rehearsal (duplicated rows) — diff skipped")
elif REF_PRED is None:
    print("no reference predictions attached — diff skipped "
          f"(attach the submitted entry CSV as one of {REFERENCE_PRED_NAMES})")
elif not IS_PUBLIC_RERUN:
    print(f"test-set ids differ from the reference ({len(T)} rows vs "
          f"{len(REF_PRED)} reference rows) -> this is the HELD-OUT run; "
          "the reproduction diff is not applicable")
else:
    mism = [(str(r["id"]).strip(), layer_of[i], REF_PRED[str(r["id"]).strip()], pred[i])
            for i, r in enumerate(T) if pred[i] != REF_PRED[str(r["id"]).strip()]]
    if not mism:
        print(f"REPRODUCTION PASS: n_mismatches = 0 — all {len(T)} predictions are "
              f"identical to the submitted Phase-1 final_entry CSV")
    else:
        print(f"REPRODUCTION FAIL: n_mismatches = {len(mism)}/{len(T)}")
        print(f"  by layer: {dict(sorted(collections.Counter(m[1] for m in mism).items()))}")
        for rid, lay, want, got in mism[:50]:
            print(f"  id={rid} layer={lay} reference={want} got={got}")
        if len(mism) > 50:
            print(f"  ... and {len(mism)-50} more")
        print("  NOTE: on the artifact fast path every layer is a deterministic "
              "file read, so a nonzero count means an artifact is missing, stale, "
              "or a later-build-only file leaked into the router (see EXCLUDED_ARTIFACTS).")
